# Crawl theo độ ưu tiên: 21 domain tiếng Việt còn thiếu (Kaggle, có checkpoint/resume)

Đo ngày 28/10/2026: corpus của nhóm mới phủ **52%** link BTC. Toàn bộ 1.200 truy vấn là tiếng Việt, và khoảng **71%** link có slug khớp truy vấn nhất nằm ở các link chưa crawl. Riêng 3 domain nhathuoclongchau, suckhoedoisong và thanhnien chiếm 54% số đó. Notebook này crawl **21 domain** đó (khoảng 380 nghìn link) theo **thứ tự ưu tiên**:

- Với mỗi truy vấn, notebook xếp link theo độ gần giữa slug URL (tức tiêu đề bài) và truy vấn (TF-IDF, bỏ dấu).
- Link nào đứng hạng cao ở bất kỳ truy vấn nào sẽ được crawl trước.
- Nhờ vậy, dù phải dừng sớm, mỗi truy vấn vẫn đã có các bài khớp nhất.

**Hai luồng chạy song song:**

- **Tải trực tiếp:** mỗi domain có một cổng giãn cách riêng, tối đa khoảng 1 request/giây/domain, hoặc theo `Crawl-delay` nếu robots.txt quy định. robots.txt được kiểm tra ở cả URL gốc lẫn mọi bước chuyển hướng.
- **Wayback Machine** (bản lưu `id_` trên web.archive.org): giãn cách 2–16 giây.
  - Dùng cho các domain trả 403 hoặc trang kiểm tra bot cho crawler (đo 28/10): nhathuoclongchau, youmed, baodanang, baohaiphong, laodong, thaythuocvietnam.
  - Domain khác bị chặn **15 lần liên tiếp** khi đang chạy sẽ **tự chuyển** phần còn lại sang Wayback.
  - Notebook **không giả danh trình duyệt và không vượt trang kiểm tra bot**.
  - Lấy mẫu 30 URL/domain ngày 28/10, tỉ lệ có bản lưu: nhathuoclongchau 97%, youmed 100%, thaythuocvietnam 90%, laodong 63%; baonghean, baodanang, baohaiphong chỉ 7–10%. Ba domain cuối được xếp cuối hàng Wayback.

**Đầu ra** giống hệt schema các shard đã có:

- `corpus/viz/docs/a/viz-a-direct-NNNNN.jsonl.zst` và `viz-a-wayback-NNNNN.jsonl.zst` (10.000 document mỗi shard), kèm manifest và crawl log.
- `doc_id` là id gốc của BTC.
- Các id trong những domain này mà nhóm đã có thì bỏ qua. Danh sách gồm cả shard `viz-b-00026` đến `viz-b-00028` của bạn B (cập nhật 29/10): vov2, vietnamplus, tienphong, thaythuocvietnam. Khi resume từ checkpoint cũ, các URL còn chờ trùng id cũng được bỏ qua.

## Cách chạy trên Kaggle

1. **Settings → Internet: On**. Accelerator chọn None (CPU là đủ).
2. **Add Data (bắt buộc để xếp ưu tiên):** tạo một Dataset chứa file `queries/query.parquet` của repo rồi đính kèm. Thiếu file này thì notebook vẫn chạy, nhưng chỉ xếp xen kẽ theo domain nên kém hiệu quả hơn nhiều.
3. (Tuỳ chọn) đính kèm `links_corpus_vi_en_zh.parquet`. Không có thì notebook tự tải từ Hugging Face và kiểm tra SHA-256.
4. Chạy bằng **Save Version → Save & Run All**.
   - Notebook dừng mềm sau `MAX_RUNTIME_HOURS` (11 giờ), xuất shard và đóng gói `prio_resume.zip`.
   - Bundle này cũng được làm mới mỗi 60 phút.
5. **Resume:** tạo version mới, **Add Data → Notebook Output** của version trước, kèm Dataset `query.parquet` như cũ.
   - Notebook tự nhận `prio_resume.zip` và chạy tiếp đúng chỗ dừng.
   - Thứ tự ưu tiên và chỉ mục CDX được giữ nguyên, không tính lại.

**Thử nhanh:** thêm cell `os.environ["CRAWL_LIMIT"] = "3"` trước cell cấu hình. Mỗi domain và luồng Wayback chỉ tải 3 URL.

## Đưa kết quả về máy local

Tải Output về, rồi chép `corpus/viz/docs/a/*`, `corpus/viz/manifest/a/*` và `corpus/viz/crawl_log/a/*` vào cùng đường dẫn trong repo. Tên shard `viz-a-direct-*` và `viz-a-wayback-*` không trùng shard nào đang có.

Lưu ý khi resume: shard của version trước nằm sẵn trong `prio_resume.zip` và trong Output. Version mới chỉ thêm shard có số thứ tự lớn hơn, nên chỉ cần chép những file mới.

Sau đó chạy `python -m retrieval.run_index`, rồi `python -m retrieval.run_queries --fresh`.

In [ ]:
%pip -q install "trafilatura==2.3.1" "langdetect==1.0.9" "protego>=0.3" "zstandard>=0.22" "aiohttp>=3.9" "pyarrow>=14"

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import json
import os
import re
import shutil
import sqlite3
import time
import unicodedata
import zipfile
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from email.utils import parsedate_to_datetime
from pathlib import Path, PurePosixPath
from urllib.parse import unquote, urljoin, urlsplit
from urllib.request import Request, urlopen

import aiohttp
import pyarrow as pa
import pyarrow.parquet as pq
import trafilatura
import zstandard
from protego import Protego

RUN_STARTED = time.monotonic()

OWNER = "a"
SCHEMA_VERSION = "1.0"
NORM_VERSION = "n1"
CODE_COMMIT = "kaggle-priority-v1"
USER_AGENT = "R2AI2026CorpusResearch/1.0 (respectful research crawler)"

LINKS_FILENAME = "links_corpus_vi_en_zh.parquet"
LINKS_SHA256 = "87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7"
LINKS_ROWS = 1_220_598
LINKS_HF_URL = (
    "https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/"
    "resolve/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet?download=true"
)
QUERIES_FILENAME = "query.parquet"          # đính kèm queries/query.parquet của repo để xếp ưu tiên theo truy vấn

# 21 domain tiếng Việt mà corpus của nhóm chưa có (đo 28/10/2026: phủ < 1%). Hostname đã bỏ "www.".
TARGET_DOMAINS = (
    "suckhoedoisong.vn", "nhathuoclongchau.com.vn", "thanhnien.vn", "laodong.vn", "baohaiphong.vn", "baonghean.vn",
    "baodanang.vn", "hellobacsi.com", "vov.vn", "khoahocphothong.vn", "youmed.vn", "vov2.vov.vn", "qdnd.vn",
    "vietnamplus.vn", "tienphong.vn", "thaythuocvietnam.vn", "dantri.com.vn", "baolangson.vn", "hanoimoi.vn",
    "benhviennhitrunguong.gov.vn", "baotayninh.vn",
)
# Đo 28/10/2026 bằng User-Agent thật của notebook: trả 403, trang kiểm tra bot, robots.txt không đọc được, hoặc bài đã
# chuyển về trang chủ → lấy bản lưu Wayback ngay. Domain khác bị chặn khi chạy cũng tự chuyển sang Wayback.
WAYBACK_FIRST = {"nhathuoclongchau.com.vn", "youmed.vn", "baodanang.vn", "baohaiphong.vn", "laodong.vn",
                 "thaythuocvietnam.vn"}
# Lấy mẫu 30 URL/domain ngày 28/10: Wayback có bản lưu 97% longchau, 100% youmed, 90% thaythuocvietnam, 63% laodong nhưng
# chỉ ~7–10% ở 3 domain dưới → xếp cuối hàng Wayback để không tốn lượt cho URL không có bản lưu.
WAYBACK_LAST = {"baodanang.vn", "baohaiphong.vn", "baonghean.vn"}
# Đã có trong shard của nhóm (gồm viz-b-00026..00028 của bạn B, 29/10: vov2, vietnamplus, tienphong, thaythuocvietnam):
# không crawl lại; khi resume, URL còn chờ có id trong danh sách này được đánh dấu bỏ qua.
ALREADY_CRAWLED_IDS = frozenset(json.loads("[119029,119146,119211,119318,119356,120247,120377,120623,120697,120717,120732,120764,120868,120904,120946,120979,121104,121556,121670,121824,121981,122354,122493,122502,122703,122800,122854,122965,123034,123106,123163,123192,123661,123709,123736,124184,124268,124462,124660,124698,125195,125240,125408,125413,125934,126104,126118,126146,126171,126230,126378,126425,126482,126520,126578,126583,126654,126675,126677,126696,126716,126838,127051,127096,127111,127244,127397,127693,588667,599663,620725,628799,630141,634670,644465,659731,659825,659863,659917,660069,660077,660108,660188,660197,660407,660438,660621,661064,661152,661160,661592,661803,661823,661830,661831,661832,661833,661834,661835,661838,661839,661841,661842,661845,661846,661847,661849,661850,661851,661852,661853,661854,661855,661856,661857,661858,661859,661860,661861,661862,661868,661871,661875,661877,661878,661879,661881,661882,661883,661884,661885,661887,661888,661889,661890,661891,661892,661893,661894,661895,661896,661897,661898,661899,661900,661901,661902,661903,661904,661905,661906,661907,661908,661909,661910,661911,661912,661913,661914,661915,661916,661917,661918,661919,661920,661921,661922,661923,661924,661925,661926,661927,661928,661929,661930,661931,661932,661933,661934,661935,661936,661937,661938,661939,661940,661941,661942,661943,661944,661945,661946,661947,661948,661949,661950,661951,661952,661953,661954,661955,661956,661957,661958,661959,661960,661961,661962,661963,661964,661965,661966,661967,661968,661969,661970,661971,661972,661973,661974,661975,661976,661977,661978,661979,661980,661981,661982,661983,661984,661985,661986,661987,661988,661989,661990,661991,661992,661993,661994,661995,661996,661997,661998,661999,662000,662001,662002,662003,662004,662005,662006,662007,662008,662009,662010,662011,662012,662013,662014,662015,662016,662017,662018,662019,662020,662021,662022,662023,662024,662025,662026,662027,662028,662029,662030,662031,662032,662033,662034,662035,662036,662037,662038,662039,662040,662041,662042,662043,662044,662045,662046,662047,662048,662049,662050,662051,662052,662054,662055,662056,662058,662059,662060,662061,662062,662063,662064,662065,662066,662067,662068,662069,662070,662071,662072,662073,662074,662075,662076,662077,675772,675773,675774,675775,675776,675777,675778,675779,675780,675781,675782,675783,675784,675785,675786,675787,675788,675789,675790,675791,675792,675793,675794,675795,675796,675797,675798,675799,675800,675801,675802,675803,675804,675805,675806,675807,675808,675809,675810,675811,675812,675813,675814,675815,675816,675817,675818,675819,675820,675821,675822,675823,675824,675825,675826,675827,675828,675829,675830,675831,675832,675833,675834,675835,675836,675837,675838,675839,675840,675841,675842,675843,675844,675845,675846,675847,675848,675849,675850,675851,675852,675853,675854,675855,675856,675857,675858,675859,675860,675861,675862,675863,675864,675865,675866,675867,675868,675869,675870,675871,675872,675873,675874,675875,675876,675877,675878,675879,675880,675881,675882,675883,675884,675885,675886,675887,675888,675889,675890,675891,675892,675893,675894,675895,675896,675897,675898,675899,675900,675901,675902,675903,675904,675905,675906,675907,675908,675909,675910,675911,675912,675913,675914,675915,675916,675917,675918,675919,675920,675921,675922,675923,675924,675925,675926,675927,675928,675929,675930,675931,675932,675933,675934,675935,675936,675937,675938,675939,675940,675941,675942,675943,675944,675945,675946,675947,675948,675949,675950,675951,675952,675953,675954,675955,675956,675957,675958,675959,675960,675961,675962,675963,675964,675965,675966,675967,675968,675969,675970,675971,675972,675973,675974,675975,675976,675977,675978,675979,675980,675981,675982,675983,675984,675985,675986,675987,675988,675989,675990,675991,675992,675993,675994,675995,675996,675997,675998,675999,676000,676001,676002,676003,676004,676005,676006,676007,676008,676009,676010,676011,676012,676013,676014,676015,676016,676017,676018,676019,676020,676021,676022,676023,676024,676025,676026,676027,676028,676029,676030,676031,676032,676033,676034,676035,676036,676037,676038,676039,676040,676041,676042,676043,676044,676045,676046,676047,676048,676049,676050,676051,676052,676053,676054,676055,676056,676057,676058,676059,676060,676061,676062,676063,676064,676065,676066,676067,676068,676069,676070,676071,676072,676073,676074,676075,676076,676077,676078,676079,676080,676081,676082,676083,676084,676085,676086,676087,676088,676089,676090,676091,676092,676093,676094,676095,676096,676097,676098,676099,676100,676101,676102,676103,676104,676105,676106,676107,676108,676109,676110,676111,676112,676113,676114,676115,676116,676117,676118,676119,676120,676121,676122,676123,676124,676125,676126,676127,676128,676129,676130,676131,676132,676133,676134,676135,676136,676137,676138,676139,676140,676141,676142,676143,676144,676145,676146,676147,676148,676149,676150,676151,676152,676153,676154,676155,676156,676157,676158,676159,676160,676161,676162,676163,676164,676165,676166,676167,676168,676169,676170,676171,676172,676173,676174,676175,676176,676177,676178,676179,676180,676181,676182,676183,676184,676185,676186,676187,676188,676189,676190,676191,676192,676193,676194,676195,676196,676197,676198,676199,676200,676201,676202,676203,676204,676205,676206,676207,676208,676209,676210,676211,676212,676213,676214,676215,676216,676217,676218,676219,676220,676221,676222,676223,676224,676225,676226,676227,676228,676229,676230,676231,676232,676233,676234,676235,676236,676237,676238,676239,676240,676241,676242,676243,676244,676245,676246,676247,676248,676249,676250,676251,676252,676253,676254,676255,676256,676257,676258,676259,676260,676261,676262,676263,676264,676265,676266,676267,676268,676269,676270,676271,676272,676273,676274,676275,676276,676277,676278,676279,676280,676281,676282,676283,676284,676285,676286,676287,676288,676289,676290,676291,676292,676293,676294,676295,676296,676297,676298,676299,676300,676301,676302,676303,676304,676305,676306,676307,676308,676309,676310,676311,676312,676313,676314,676315,676316,676317,676318,676319,676320,676321,676322,676323,676324,676325,676326,676327,676328,676329,676330,676331,676332,676333,676334,676335,676336,676337,676338,676339,676340,676341,676342,676343,676344,676345,676346,676347,676348,676349,676350,676351,676352,676353,676354,676355,676356,676357,676358,676359,676360,676361,676362,676363,676364,676365,676366,676367,676368,676369,676370,676371,676372,676373,676374,676375,676376,676377,676378,676379,676380,676381,676382,676383,676384,676385,676386,676387,676388,676389,676390,676391,676392,676393,676394,676395,676396,676397,676398,676399,676400,676401,676402,676403,676404,676405,676406,676407,676408,676409,676410,676411,676412,676413,676414,676415,676416,676417,676418,676419,676420,676421,676422,676423,676424,676425,676426,676427,676428,676429,676430,676431,676432,676433,676434,676435,676436,676437,676438,676439,676440,676441,676442,676443,676444,676445,676446,676447,676448,676449,676450,676451,676452,676453,676454,676455,676456,676457,676458,676459,676460,676461,676462,676463,676464,676465,676466,676467,676468,676469,676470,676471,676472,676473,676474,676475,676476,676477,676478,676479,676480,676481,676482,676483,676484,676485,676486,676487,676488,676489,676490,676491,676492,676493,676494,676495,676496,676497,676498,676499,676500,676501,676502,676503,676504,676505,676506,676507,676508,676509,676510,676511,676512,676513,676514,676515,676516,676517,676518,676519,676520,676521,676522,676523,676524,676525,676526,676527,676528,676529,676530,676531,676532,676533,676534,676535,676536,676537,676538,676539,676540,676541,676542,676543,676544,676545,676546,676547,676548,676549,676550,676551,676552,676553,676554,676555,676556,676557,676558,676559,676560,676561,676562,676563,676564,676565,676566,676567,676568,676569,676570,676571,676572,676573,676574,676575,676576,676577,676578,676579,676580,676581,676582,676583,676584,676585,676586,676587,676588,676589,676590,676591,676592,676593,676594,676595,676596,676597,676598,676599,676600,676601,676602,676603,676604,676605,676606,676607,676608,676609,676610,676611,676612,676613,676614,676615,676616,676617,676618,676619,676620,676621,676622,676623,676624,676625,676626,676627,676628,676629,676630,676631,676632,676633,676634,676635,676636,676637,676638,676639,676640,676641,676642,676643,676644,676645,676646,676647,676648,676649,676650,676651,676652,676653,676654,676655,676656,676657,676658,676659,676660,676661,676662,676663,676664,676665,676666,676667,676668,676669,676670,676671,676672,676673,676674,676675,676676,676677,676678,676679,676680,676681,676682,676683,676684,676685,676686,676687,676688,676689,676690,676691,676692,676693,676694,676695,676696,676697,676698,676699,676700,676701,676702,676703,676704,676705,676706,676707,676708,676709,676710,676711,676712,676713,676714,676715,676716,676717,676718,676719,676720,676721,676722,676723,676724,676725,676726,676727,676728,676729,676730,676731,676732,676733,676734,676735,676736,676737,676738,676739,676740,676741,676742,676743,676744,676745,676746,676747,676748,676749,676750,676751,676752,676753,676754,676755,676756,676757,676758,676759,676760,676761,676762,676763,676764,676765,676766,676767,676768,676769,676770,676771,676772,676773,676774,676775,676776,676777,676778,676779,676780,676781,676782,676783,676784,676785,676786,676787,676788,676789,676790,676791,676792,676793,676794,676795,676796,676797,676798,676799,676800,676801,676802,676803,676804,676805,676806,676807,676808,676809,676810,676811,676812,676813,676814,676815,676816,676817,676818,676819,676820,676821,676822,676823,676824,676825,676826,676827,676828,676829,676830,676831,676832,676833,676834,676835,676836,676837,676838,676839,676840,676841,676842,676843,676844,676845,676846,676847,676848,676849,676850,676851,676852,676853,676854,676855,676856,676857,676858,676859,676860,676861,676862,676863,676864,676865,676866,676867,676868,676869,676870,676871,676872,676873,676874,676875,676876,676877,676878,676879,676880,676881,676882,676883,676884,676885,676886,676887,676888,676889,676890,676891,676892,676893,676894,676895,676896,676897,676898,676899,676900,676901,676902,676903,676904,676905,676906,676907,676908,676909,676910,676911,676912,676913,676914,676915,676916,676917,676918,676919,676920,676921,676922,676923,676924,676925,676926,676927,676928,676929,676930,676931,676932,676933,676934,676935,676936,676937,676938,676939,676940,676941,676942,676943,676944,676945,676946,676947,676948,676949,676950,676951,676952,676953,676954,676955,676956,676957,676958,676959,676960,676961,676962,676963,676964,676965,676966,676967,676968,676969,676970,676971,676972,676973,676974,676975,676976,676977,676978,676979,676980,676981,676982,676983,676984,676985,676986,676987,676988,676989,676990,676991,676992,676993,676994,676995,676996,676997,676998,676999,677000,677001,677002,677003,677004,677005,677006,677007,677008,677009,677010,677011,677012,677013,677014,677015,677016,677017,677018,677019,677020,677021,677022,677023,677024,677025,677026,677027,677028,677029,677030,677031,677032,677033,677034,677035,677036,677037,677038,677039,677040,677041,677042,677043,677044,677045,677046,677047,677048,677049,677050,677051,677052,677053,677054,677055,677056,677057,677058,677059,677060,677061,677062,677063,677064,677065,677066,677067,677068,677069,677070,677071,677072,677073,677074,677075,677076,677077,677078,677079,677080,677081,677082,677083,677084,677085,677086,677087,677088,677089,677090,677091,677092,677093,677094,677095,677096,677097,677098,677099,677100,677101,677102,677103,677104,677105,677106,677107,677108,677109,677110,677111,677112,677113,677114,677115,677116,677117,677118,677119,677120,677121,677122,677123,677124,677125,677126,677127,677128,677129,677130,677131,677132,677133,677134,677135,677136,677137,677138,677139,677140,677141,677142,677143,677144,677145,677146,677147,677148,677149,677150,677151,677152,677153,677154,677155,677156,677157,677158,677159,677160,677161,677162,677163,677164,677165,677166,677167,677168,677169,677170,677171,677172,677173,677174,677175,677176,677177,677178,677179,677180,677181,677182,677183,677184,677185,677186,677187,677188,677189,677190,677191,677192,677193,677194,677195,677196,677197,677198,677199,677200,677201,677202,677203,677204,677205,677206,677207,677208,677209,677210,677211,677212,677213,677214,677215,677216,677217,677218,677219,677220,677221,677222,677223,677224,677225,677226,677227,677228,677229,677230,677231,677232,677233,677234,677235,677236,677237,677238,677239,677240,677241,677242,677243,677244,677245,677246,677247,677248,677249,677250,677251,677252,677253,677254,677255,677256,677257,677258,677259,677260,677261,677262,677263,677264,677265,677266,677267,677268,677269,677270,677271,677272,677273,677274,677275,677276,677277,677278,677279,677280,677281,677282,677283,677284,677285,677286,677287,677288,677289,677290,677291,677292,677293,677294,677295,677296,677297,677298,677299,677300,677301,677302,677303,677304,677305,677306,677307,677308,677309,677310,677311,677312,677313,677314,677315,677316,677317,677318,677319,677320,677321,677322,677323,677324,677325,677326,677327,677328,677329,677330,677331,677332,677333,677334,677335,677336,677337,677338,677339,677340,677341,677342,677343,677344,677345,677346,677347,677348,677349,677350,677351,677352,677353,677354,677355,677356,677357,677358,677359,677360,677361,677362,677363,677364,677365,677366,677367,677368,677369,677370,677371,677372,677373,677374,677375,677376,677377,677378,677379,677380,677381,677382,677383,677384,677385,677386,677387,677388,677389,677390,677391,677392,677393,677394,677395,677396,677397,677398,677399,677400,677401,677402,677403,677404,677405,677406,677407,677408,677409,677410,677411,677412,677413,677414,677415,677416,677417,677418,677419,677420,677421,677422,677423,677424,677425,677426,677427,677428,677429,677430,677431,677432,677433,677434,677435,677436,677437,677438,677439,677440,677441,677442,677443,677444,677445,677446,677447,677448,677449,677450,677451,677452,677453,677454,677455,677456,677457,677458,677459,677460,677461,677462,677463,677464,677465,677466,677467,677468,677469,677470,677471,677472,677473,677474,677475,677476,677477,677478,677479,677480,677481,677482,677483,677484,677485,677486,677487,677488,677489,677490,677491,677492,677493,677494,677495,677496,677497,677498,677499,677500,677501,677502,677503,677504,677505,677506,677507,677508,677509,677510,677511,677512,677513,677514,677515,677516,677517,677518,677519,677520,677521,677522,677523,677524,677525,677526,677527,677528,677529,677530,677531,677532,677533,677534,677535,677536,677537,677538,677539,677540,677541,677542,677543,677544,677545,677546,677547,677548,677549,677550,677551,677552,677553,677554,677555,677556,677557,677558,677559,677560,677561,677562,677563,677564,677565,677566,677567,677568,677569,677570,677571,677572,677573,677574,677575,677576,677577,677578,677579,677580,677581,677582,677583,677584,677585,677586,677587,677588,677589,677590,677591,677592,677593,677594,677595,677596,677597,677598,677599,677600,677601,677602,677603,677604,677605,677606,677607,677608,677609,677610,677611,677612,677613,677614,677615,677616,677617,677618,677619,677620,677621,677622,677623,677624,677625,677626,677627,677628,677629,677630,677631,677632,677633,677634,677635,677636,677637,677638,677639,677640,677641,677642,677643,677644,677645,677646,677647,677648,677649,677650,677651,677652,677653,677654,677655,677656,677657,677658,677659,677660,677661,677662,677663,677664,677665,677666,677667,677668,677669,677670,677671,677672,677673,677674,677675,677676,677677,677678,677679,677680,677681,677682,677683,677684,677685,677686,677687,677688,677689,677690,677691,677692,677693,677694,677695,677696,677697,677698,677699,677700,677701,677702,677703,677704,677705,677706,677707,677708,677709,677710,677711,677712,677713,677714,677715,677716,677717,677718,677719,677720,677721,677722,677723,677724,677725,677726,677727,677728,677729,677730,677731,677732,677733,677734,677735,677736,677737,677738,677739,677740,677741,677742,677743,677744,677745,677746,677747,677748,677749,677750,677751,677752,677753,677754,677755,677756,677757,677758,677759,677760,677761,677762,677763,677764,677765,677766,677767,677768,677769,677770,677771,677772,677773,677774,677775,677776,677777,677778,677779,677780,677781,677782,677783,677784,677785,677786,677787,677788,677789,677790,677791,677792,677793,677794,677795,677796,677797,677798,677799,677800,677801,677802,677803,677804,677805,677806,677807,677808,677809,677810,677811,677812,677813,677814,677815,677816,677817,677818,677819,677820,677821,677822,677823,677824,677825,677826,677827,677828,677829,677830,677831,677832,677833,677834,677835,677836,677837,677838,677839,677840,677841,677842,677843,677844,677845,677846,677847,677848,677849,677850,677851,677852,677853,677854,677855,677856,677857,677858,677859,677860,677861,677862,677863,677864,677865,677866,677867,677868,677869,677870,677871,677872,677873,677874,677875,677876,677877,677878,677879,677880,677881,677882,677883,677884,677885,677886,677887,677888,677889,677890,677891,677892,677893,677894,677895,677896,677897,677898,677899,677900,677901,677902,677903,677904,677905,677906,677907,677908,677909,677910,677911,677912,677913,677914,677915,677916,677917,677918,677919,677920,677921,677922,677923,677924,677925,677926,677927,677928,677929,677930,677931,677932,677933,677934,677935,677936,677937,677938,677939,677940,677941,677942,677943,677944,677945,677946,677947,677948,677949,677950,677951,677952,677953,677954,677955,677956,677957,677958,677959,677960,677961,677962,677963,677964,677965,677966,677967,677968,677969,677970,677971,677972,677973,677974,677975,677976,677977,677978,677979,677980,677981,677982,677983,677984,677985,677986,677987,677988,677989,677990,677991,677992,677993,677994,677995,677996,677997,677998,677999,678000,678001,678002,678003,678004,678005,678006,678007,678008,678009,678010,678011,678012,678013,678014,678015,678016,678017,678018,678019,678020,678021,678022,678023,678024,678025,678026,678027,678028,678029,678030,678031,678032,678033,678034,678035,678036,678037,678038,678039,678040,678041,678042,678043,678044,678045,678046,678047,678048,678049,678050,678051,678052,678053,678054,678055,678056,678057,678058,678059,678060,678061,678062,678063,678064,678065,678066,678067,678068,678069,678070,678071,678072,678073,678074,678075,678076,678077,678078,678079,678080,678081,678082,678083,678084,678085,678086,678087,678088,678089,678090,678091,678092,678093,678094,678095,678096,678097,678098,678099,678100,678101,678102,678103,678104,678105,678106,678107,678108,678109,678110,678111,678112,678113,678114,678115,678116,678117,678118,678119,678120,678121,678122,678123,678124,678125,678126,678127,678128,678129,678130,678131,678132,678133,678134,678135,678136,678137,678138,678139,678140,678141,678142,678143,678144,678145,678146,678147,678148,678149,678150,678151,678152,678153,678154,678155,678156,678157,678158,678159,678160,678161,678162,678163,678164,678165,678166,678167,678168,678169,678170,678171,678172,678173,678174,678175,678176,678177,678178,678179,678180,678181,678182,678183,678184,678185,678186,678187,678188,678189,678190,678191,678192,678193,678194,678195,678196,678197,678198,678199,678200,678201,678202,678203,678204,678205,678206,678207,678208,678209,678210,678211,678212,678213,678214,678215,678216,678217,678218,678219,678220,678221,678222,678223,678224,678225,678226,678227,678228,678229,678230,678231,678232,678233,678234,678235,678236,678237,678238,678239,678240,678241,678242,678243,678244,678245,678246,678247,678248,678249,678250,678251,678252,678253,678254,678255,678256,678257,678258,678259,678260,678261,678262,678263,678264,678265,678266,678267,678268,678269,678270,678271,678272,678273,678274,678275,678276,678277,678278,678279,678280,678281,678282,678283,678284,678285,678286,678287,678288,678289,678290,678291,678292,678293,678294,678295,678296,678297,678298,678299,678300,678301,678302,678303,678304,678305,678306,678307,678308,678309,678310,678311,678312,678313,678314,678315,678316,678317,678318,678319,678320,678321,678322,678323,678324,678325,678326,678327,678328,678329,678330,678331,678332,678333,678334,678335,678336,678337,678338,678339,678340,678341,678342,678343,678344,678345,678346,678347,678348,678349,678350,678351,678352,678353,678354,678355,678356,678357,678358,678359,678360,678361,678362,678363,678364,678365,678366,678367,678368,678369,678370,678371,678372,678373,678374,678375,678376,678377,678378,678379,678380,678381,678382,678383,678384,678385,678386,678387,678388,678389,678390,678391,678392,678393,678394,678395,678396,678397,678398,678399,678400,678401,678402,678403,678404,678405,678406,678407,678408,678409,678410,678411,678412,678413,678414,678415,678416,678417,678418,678419,678420,678421,678422,678423,678424,678425,678426,678427,678428,678429,678430,678431,678432,678433,678434,678435,678436,678437,678438,678439,678440,678441,678442,678443,678444,678445,678446,678447,678448,678449,678450,678451,678452,678453,678454,678455,678456,678457,678458,678459,678460,678461,678462,678463,678464,678465,678466,678467,678468,678469,678470,678471,678472,678473,678474,678475,678476,678477,678478,678479,678480,678481,678482,678483,678484,678485,678486,678487,678488,678489,678490,678491,678492,678493,678494,678495,678496,678497,678498,678499,678500,678501,678502,678503,678504,678505,678506,678507,678508,678509,678510,678511,678512,678513,678514,678515,678516,678517,678518,678519,678520,678521,678522,678523,678524,678525,678526,678527,678528,678529,678530,678531,678532,678533,678534,678535,678536,678537,678538,678539,678540,678541,678542,678543,678544,678545,678546,678547,678548,678549,678550,678551,678552,678553,678554,678555,678556,678557,678558,678559,678560,678561,678562,678563,678564,678565,678566,678567,678568,678569,678570,678571,678572,678573,678574,678575,678576,678577,678578,678579,678580,678581,678582,678583,678584,678585,678586,678587,678588,678589,678590,678591,678592,678593,678594,678595,678596,678597,678598,678599,678600,678601,678602,678603,678604,678605,678606,678607,678608,678609,678610,678611,678612,678613,678614,678615,678616,678617,678618,678619,678620,678621,678622,678623,678624,678625,678626,678627,678628,678629,678630,678631,678632,678633,678634,678635,678636,678637,678638,678639,678640,678641,678642,678643,678644,678645,678646,678647,678648,678649,678650,678651,678652,678653,678654,678655,678656,678657,678658,678659,678660,678661,678662,678663,678664,678665,678666,678667,678668,678669,678670,678671,678672,678673,678674,678675,678676,678677,678678,678679,678680,678681,678682,678683,678684,678685,678686,678687,678688,678689,678690,678691,678692,678693,678694,678695,678696,678697,678698,678699,678700,678701,678702,678703,678704,678705,678706,678707,678708,678709,678710,678711,678712,678713,678714,678715,678716,678717,678718,678719,678720,678721,678722,678723,678724,678725,678726,678727,678728,678729,678730,678731,678732,678733,678734,678735,678736,678737,678738,678739,678740,678741,678742,678743,678744,678745,678746,678747,678748,678749,678750,678751,678752,678753,678754,678755,678756,678757,678758,678759,678760,678761,678762,678763,678764,678765,678766,678767,678768,678769,678770,678771,678772,678773,678774,678775,678776,678777,678778,678779,678780,678781,678782,678783,678784,678785,678786,678787,678788,678789,678790,678791,678792,703735,703737,703744,703756,703758,703762,703789,703797,703801,703808,703813,703824,703825,703829,703834,703837,703842,703871,703879,703886,703892,703893,703896,703897,703900,703902,703903,703904,703905,703913,703936,703952,703958,703961,703967,703970,703972,703975,703989,703990,703993,703994,703996,704005,713526,713534,713640,713676,713910,713911,713912,713913,713914,713915,713916,713917,713918,713919,713920,713946,713947,713948,713949,713950,713951,713952,713953,713954,714012,714013,714014,714015,714016,714017,714018,714019,714020,714021,714023,714024,714025,714026,714027,714028,714030,714031,714032,714033,714036,714037,714038,714039,714041,714044,714045,714046,714047,714048,714049,714050,714051,714052,714053,714054,714055,714056,714057,714058,714060,714061,714062,714063,714064,714065,714066,714067,714068,714069,714070,714071,714072,714073,714074,714075,714076,714077,714078,714079,714082,714083,714085,714086,714087,714088,714089,714090,714091,714092,714093,714094,714095,714096,714097,714098,714099,714100,714101,714102,714104,714105,714106,714107,714108,714109,714110,714111,714113,714114,714116,714117,714118,714119,714120,714121,714123,714127,714128,714129,714131,714132,714133,714135,714137,714138,714139,714140,714141,714142,714143,714144,714145,714146,714147,714148,714149,714151,714152,714153,714157,714159,714162,714163,714164,714165,714168,714169,714170,714171,714172,714173,714174,714175,714176,714177,714178,714179,714180,714181,714183,714184,714185,714186,714188,714189,714190,714191,714192,714193,714195,714198,714199,714200,714202,714203,714204,714205,714206,714207,714208,714209,714210,714212,714213,714214,714215,714216,714218,714219,714220,714221,714226,714227,714228,714229,714230,714231,714232,714235,714236,714237,714238,714239,714240,714241,714242,714243,714244,714245,714247,714248,714249,714252,714253,714254,714256,714259,714260,714261,714263,714264,714265,714266,714267,714268,714271,714274,714275,714276,714277,714278,714279,714281,714282,714283,714284,714285,714286,714287,714288,714291,714292,714293,714294,714295,714296,714297,714298,714299,714300,714301,714302,714303,714304,714306,714307,714309,714310,714311,714312,714313,714314,714315,714316,714317,714318,714320,714321,714322,714323,714324,714325,714326,714327,714328,714329,714330,714331,714332,714333,714334,714335,714336,714338,714339,714340,714341,714342,714343,714344,714345,714346,714347,714350,714351,714352,714353,714354,714355,714356,714357,714358,714359,714360,714361,714362,714363,714364,714365,714366,714367,714368,714369,714370,714371,714372,714373,714374,714375,714376,714377,714378,714379,714380,714382,714383,714384,714385,714386,714387,714388,714389,714390,714391,714392,714393,714394,714395,714396,714398,714400,714401,714402,714404,714405,714406,714408,714409,714410,714411,714412,714413,714414,714415,714417,714419,714420,714421,714422,714423,714424,714425,714426,714427,714429,714430,714431,714432,714433,714434,714436,714437,714438,714439,714440,714441,714442,714443,714444,714445,714446,714447,714448,714449,714450,714451,714452,714454,714455,714456,714457,714459,714460,714461,714466,714467,714468,714469,714470,714471,714472,714473,714474,714475,714476,714477,714479,714481,714482,714483,714484,714485,714486,714487,714488,714489,714490,714491,714492,714493,714494,714495,714496,714497,714498,714499,714500,714501,714502,714504,714505,714506,714507,714508,714509,714510,714511,714512,714513,714515,714516,714517,714518,714521,714522,714523,714524,714525,714526,714527,714528,714530,714532,714533,714534,714535,714536,714537,714538,714539,714540,714542,714543,714544,714545,714546,714547,714548,714549,714550,714552,714553,714554,714555,714556,714557,714558,714560,714562,714563,714564,714565,714567,714568,714569,714570,714572,714573,714574,714575,714576,714577,714578,714579,714580,714581,714582,714583,714584,714585,714586,714588,714590,714591,714592,714593,714594,714595,714596,714597,714598,714599,714601,714602,714603,714605,714606,714607,714608,714609,714610,714611,714612,714613,714614,714615,714617,714618,714619,714620,714621,714622,714623,714624,714625,714626,714628,714629,714630,714631,714632,714633,714634,714635,714636,714637,714638,714639,714640,714642,714643,714644,714645,714646,714647,714649,714650,714651,714652,714654,714655,714656,714657,714659,714660,714661,714662,714663,714664,714665,714666,714667,714668,714669,714670,714671,714672,714673,714674,714675,714676,714677,714678,714680,714681,714682,714683,714684,714685,714686,714687,714688,714689,714690,714691,714692,714693,714694,714696,714697,714698,714699,714700,714701,714702,714703,714704,714705,714706,714708,714709,714710,714711,714713,714714,714715,714716,714717,714718,714719,714720,714721,714723,714724,714725,714727,714728,714729,714730,714731,714732,714733,714734,714735,714737,714738,714739,714740,714741,714742,714743,714744,714745,714746,714747,714748,714749,714750,714751,714752,714753,714755,714756,714757,714758,714759,714760,714761,714762,714764,714766,714767,714769,714770,714771,714772,714773,714774,714775,714776,714777,714778,714779,714780,714781,714782,714783,714784,714785,714786,714787,714788,714789,714790,714791,714793,714794,714795,714803,714804,714805,714806,714807,714808,714809,714810,714811,714812,714813,714815,714816,714817,714819,714820,714821,714823,714824,714825,714826,714827,714828,714830,714831,714832,714833,714834,714835,714837,714839,714841,714842,714843,714844,714848,714850,714851,714852,714854,714856,714858,714860,714861,714862,714863,714864,714865,714866,714867,714868,714870,714871,714872,714873,714874,714875,714877,714878,714879,714880,714881,714882,714883,714886,714887,714891,714892,714893,714894,714895,714896,714897,714898,714899,714900,714901,714903,714904,714905,714906,714908,714909,714910,714911,714912,714914,714915,714916,714917,714918,714919,714920,714921,714922,714923,714924,714925,714926,714927,714928,714929,714930,714932,714933,714934,714935,714936,714937,714939,714940,714941,714942,714944,714946,714947,714949,714950,714951,714952,714953,714954,714955,714956,714957,714959,714960,714961,714962,714964,714966,714967,714969,714970,714971,714973,714975,714977,714978,714979,714981,714982,714983,714984,714985,714986,714987,714988,714989,714990,714992,714993,714994,714995,714996,714998,714999,715000,715001,715002,715003,715006,715007,715008,715009,715010,715011,715012,715013,715016,715018,715019,715021,715023,715024,715026,715027,715028,715030,715032,715033,715034,715035,715036,715037,715039,715040,715041,715043,715044,715045,715047,715048,715049,715050,715051,715052,715054,715055,715057,715058,715059,715060,715062,715063,715064,715065,715066,715067,715068,715069,715070,715072,715073,715074,715075,715076,715081,715082,715083,715085,715086,715090,715091,715092,715094,715096,715097,715098,715099,715101,715103,715104,715109,715110,715111,715112,715116,715117,715118,715119,715120,715121,715122,715123,715124,715125,715128,715129,715130,715131,715132,715133,715134,715135,715136,715140,715141,715142,715143,715144,715146,715147,715148,715151,715155,715157,715158,715159,715160,715161,715162,715163,715167,715168,715169,715175,715176,715177,715178,715179,715181,715184,715188,715194,715195,715197,715198,715199,715200,715201,715203,715208,715211,715212,715214,715215,715217,715218,715222,715223,715224,715225,715226,715227,715228,715229,715231,715235,715236,715238,715239,715243,715244,715245,715246,715247,715248,715250,715251,715252,715253,715254,715256,715257,715258,715259,715264,715265,715268,715269,715271,715272,715274,715275,715277,715278,715279,715281,715283,715284,715285,715289,715290,715292,715293,715294,715297,715298,715299,715301,715302,715303,715305,715306,715307,715309,715310,715314,715315,715316,715318,715319,715320,715321,715322,715325,715327,715329,715330,715331,715332,715333,715335,715336,715337,715338,715339,715341,715342,715343,715346,715347,715348,715349,715350,715351,715354,715355,715356,715358,715359,715361,715364,715365,715366,715367,715368,715370,715371,715372,715373,715374,715376,715377,715378,715379,715380,715381,715383,715384,715385,715387,715388,715389,715390,715391,715392,715393,715394,715397,715398,715399,715400,715402,715405,715406,715407,715408,715409,715410,715411,715413,715415,715417,715418,715422,715423,715424,715426,715427,715428,715430,715431,715432,715434,715435,715436,715437,715438,715439,715440,715442,715443,715444,715445,715446,715447,715448,715449,715450,715451,715452,715453,715456,715457,715459,715462,715463,715464,715465,715466,715467,715470,715471,715472,715473,715474,715476,715477,715478,715479,715480,715481,715482,715483,715484,715485,715486,715487,715488,715489,715490,715493,715496,715497,715498,715500,715501,715502,715503,715504,715505,715506,715507,715511,715512,715513,715514,715515,715516,715517,715518,715521,715522,715523,715524,715526,715529,715530,715531,715532,715533,715534,715535,715536,715538,715539,715540,715541,715542,715543,715544,715545,715547,715548,715549,715551,715552,715553,715554,715557,715560,715561,715563,715569,715570,715571,715575,715576,715577,715578,715579,715580,715581,715582,715583,715584,715585,715586,715587,715588,715589,715590,715591,715592,715594,715595,715597,715600,715601,715602,715603,715604,715605,715606,715607,715608,715609,715610,715611,715613,715614,715615,715616,715618,715619,715620,715621,715622,715623,715624,715625,715627,715628,715629,715632,715634,715636,715637,715638,715639,715640,715641,715642,715643,715644,715646,715648,715649,715651,715653,715654,715655,715656,715657,715658,715663,715664,715665,715666,715667,715668,715670,715672,715673,715674,715675,715676,715679,715680,715681,715682,715683,715686,715688,715689,715691,715692,715693,715694,715696,715698,715699,715700,715701,715702,715703,715704,715705,715706,715707,715710,715711,715712,715713,715714,715716,715717,715718,715719,715722,715723,715724,715725,715729,715731,715732,715733,715734,715735,715736,715737,715738,715739,715741,715742,715743,715744,715745,715746,715747,715748,715749,715750,715751,715752,715753,715754,715755,715756,715757,715758,715759,715760,715761,715762,715764,715765,715766,715767,715768,715769,715770,715772,715773,715775,715776,715778,715779,715780,715781,715782,715783,715784,715785,715786,715787,715788,715789,715790,715791,715792,715793,715794,715795,715797,715798,715799,715801,715802,715804,715805,715806,715807,715809,715810,715811,715812,715814,715815,715816,715820,715821,715822,715823,715824,715825,715829,715830,715831,715832,715833,715835,715837,715838,715839,715841,715843,715846,715847,715848,715851,715854,715855,715858,715859,715860,715861,715862,715865,715866,715867,715870,715871,715873,715874,715875,715876,715877,715878,715880,715881,715882,715883,715884,715885,715886,715887,715889,715890,715891,715892,715894,715896,715897,715898,715900,715902,715903,715904,715905,715906,715908,715909,715910,715912,715913,715914,715916,715917,715918,715919,715920,715921,715922,715923,715924,715925,715926,715927,715928,715929,715930,715931,715932,715933,715934,715935,715937,715938,715939,715940,715941,715942,715943,715944,715945,715946,715947,715948,715952,715953,715954,715955,715956,715957,715958,715959,715961,715962,715963,715964,715965,715966,715967,715969,715970,715971,715974,715975,715976,715977,715978,715979,715981,715983,715984,715985,715986,715987,715988,715989,715990,715991,715992,715993,715994,715995,715997,715998,715999,716001,716002,716003,716005,716006,716007,716008,716009,716011,716012,716014,716015,716016,716017,716018,716019,716021,716022,716023,716024,716025,716026,716028,716029,716030,716031,716034,716035,716036,716038,716039,716040,716041,716042,716043,716044,716045,716047,716048,716049,716051,716052,716053,716054,716055,716056,716057,716058,716060,716061,716062,716064,716065,716066,716067,716069,716071,716072,716073,716074,716076,716077,716078,716079,716082,716084,716085,716086,716089,716090,716091,716092,716093,716094,716096,716097,716100,716101,716102,716103,716104,716105,716106,716107,716110,716111,716112,716113,716114,716116,716117,716118,716120,716122,716124,716125,716126,716127,716128,716129,716130,716134,716135,716136,716138,716140,716141,716142,716143,716145,716147,716148,716149,716150,716151,716152,716154,716155,716156,716157,716158,716159,716160,716162,716163,716164,716165,716166,716167,716168,716169,716171,716173,716175,716176,716177,716179,716180,716183,716184,716185,716186,716188,716189,716192,716193,716194,716195,716196,716197,716198,716199,716200,716201,716202,716203,716204,716205,716206,716208,716210,716211,716212,716213,716214,716216,716217,716218,716219,716220,716221,716222,716224,716225,716226,716227,716228,716230,716233,716234,716235,716236,716237,716238,716239,716241,716242,716243,716244,716245,716246,716247,716248,716249,716251,716253,716254,716255,716256,716257,716259,716260,716261,716263,716265,716266,716267,716268,716269,716271,716273,716274,716275,716276,716277,716278,716279,716280,716281,716282,716283,716284,716286,716287,716288,716289,716291,716292,716293,716295,716296,716298,716299,716303,716304,716307,716310,716311,716312,716313,716314,716315,716316,716318,716319,716321,716322,716323,716324,716325,716326,716328,716329,716330,716331,716332,716333,716334,716335,716336,716337,716339,716340,716341,716344,716346,716347,716348,716349,716350,716351,716352,716353,716354,716355,716356,716359,716360,716362,716363,716364,716365,716366,716367,716370,716372,716373,716374,716375,716376,716377,716378,716381,716382,716383,716384,716385,716386,716387,716388,716389,716390,716391,716392,716394,716395,716396,716397,716398,716400,716401,716402,716403,716404,716405,716407,716408,716409,716410,716412,716413,716414,716415,716416,716417,716418,716420,716422,716423,716424,716426,716427,716429,716431,716432,716433,716434,716435,716436,716437,716438,716440,716441,716442,716444,716445,716447,716449,716450,716451,716452,716453,716454,716455,716456,716457,716458,716459,716461,716462,716463,716464,716467,716468,716474,716475,716476,716477,716478,716480,716482,716483,716484,716485,716486,716487,716488,716489,716490,716492,716493,716494,716495,716496,716497,716498,716501,716503,716504,716507,716508,716511,716514,716515,716516,716517,716518,716519,716521,716522,716524,716526,716527,716529,716530,716533,716534,716535,716536,716537,716539,716540,716541,716542,716543,716544,716545,716546,716547,716548,716549,716551,716553,716554,716556,716557,716559,716560,716561,716564,716566,716567,716568,716570,716572,716573,716574,716575,716576,716577,716578,716579,716580,716582,716583,716584,716587,716589,716590,716591,716592,716593,716594,716595,716596,716597,716598,716600,716602,716603,716605,716606,716607,716608,716612,716613,716614,716615,716616,716617,716618,716619,716620,716621,716622,716623,716624,716625,716626,716628,716631,716632,716633,716634,716636,716637,716638,716639,716640,716641,716642,716643,716644,716645,716647,716648,716649,716650,716651,716652,716653,716654,716655,716656,716657,716659,716660,716661,716662,716663,716664,716665,716666,716667,716669,716670,716671,716672,716673,716675,716676,716677,716678,716679,716681,716682,716683,716686,716687,716688,716689,716690,716692,716694,716695,716696,716698,716699,716701,716702,716703,716704,716705,716706,716708,716709,716711,716712,716716,716717,716718,716719,716720,716721,716722,716723,716725,716726,716729,716730,716731,716732,716733,716734,716735,716736,716737,716739,716740,716741,716742,716744,716745,716746,716747,716748,716749,716751,716752,716755,716756,716757,716758,716759,716760,716761,716762,716763,716764,716765,716767,716768,716769,716770,716771,716772,716773,716775,716776,716777,716779,716781,716782,716783,716784,716785,716786,716787,716789,716791,716792,716793,716794,716795,716796,716799,716800,716801,716802,716805,716806,716807,716808,716810,716812,716813,716814,716815,716816,716817,716819,716820,716821,716822,716823,716827,716828,716829,716830,716831,716832,716833,716834,716835,716836,716837,716838,716839,716842,716843,716844,716845,716846,716847,716848,716849,716850,716852,716853,716854,716856,716857,716859,716860,716864,716865,716866,716867,716868,716869,716870,716871,716872,716873,716874,716875,716876,716877,716878,716879,716880,716882,716883,716884,716885,716887,716888,716889,716891,716893,716894,716895,716896,716897,716898,716899,716900,716902,716903,716904,716905,716906,716907,716908,716909,716910,716912,716913,716914,716915,716916,716917,716918,716919,716920,716921,716922,716924,716925,716926,716927,716928,716929,716931,716932,716933,716934,716935,716936,716937,716939,716941,716942,716943,716944,716945,716946,716947,716949,716950,716951,716952,716953,716954,716955,716956,716958,716959,716961,716962,716963,716964,716965,716967,716968,716969,716970,716971,716972,716973,716974,716975,716976,716977,716978,716979,716980,716982,716984,716985,716986,716987,716989,716990,716991,716992,716994,716995,716996,716997,716999,717001,717002,717003,717005,717006,717007,717008,717009,717010,717012,717013,717014,717015,717017,717018,717020,717021,717022,717023,717024,717025,717026,717027,717029,717030,717031,717032,717033,717035,717036,717037,717038,717039,717040,717041,717043,717044,717045,717046,717047,717050,717051,717052,717053,717054,717055,717056,717058,717061,717065,717066,717067,717068,717069,717070,717072,717074,717075,717076,717077,717078,717079,717080,717082,717084,717085,717086,717087,717089,717091,717092,717093,717094,717096,717097,717098,717099,717100,717101,717102,717103,717104,717105,717106,717107,717108,717109,717110,717111,717112,717113,717114,717115,717117,717118,717120,717121,717122,717123,717125,717126,717127,717128,717129,717130,717131,717132,717133,717135,717136,717137,717138,717139,717141,717142,717144,717146,717148,717149,717150,717151,717153,717154,717156,717157,717158,717160,717161,717162,717163,717164,717165,717167,717170,717171,717172,717173,717176,717177,717179,717181,717184,717185,717186,717187,717189,717190,717191,717192,717193,717194,717195,717197,717199,717200,717201,717202,717203,717205,717206,717207,717208,717209,717210,717211,717212,717213,717214,717215,717216,717217,717218,717220,717221,717223,717225,717226,717227,717228,717229,717232,717233,717234,717235,717236,717237,717238,717239,717240,717241,717242,717243,717244,717245,717246,717247,717248,717249,717250,717252,717254,717256,717257,717258,717259,717260,717261,717262,717263,717264,717265,717266,717267,717268,717269,717270,717271,717272,717273,717274,717275,717276,717277,717278,717279,717281,717282,717284,717285,717286,717287,717288,717290,717291,717292,717293,717294,717295,717296,717297,717298,717299,717300,717301,717302,717304,717305,717307,717309,717310,717311,717312,717313,717314,717315,717317,717318,717319,717320,717321,717322,717323,717324,717325,717326,717328,717329,717330,717331,717332,717335,717336,717337,717338,717340,717341,717342,717343,717344,717345,717346,717347,717348,717349,717350,717351,717353,717354,717355,717356,717357,717358,717359,717360,717361,717362,717363,717364,717365,717366,717367,717368,717369,717371,717372,717373,717374,717376,717377,717378,717379,717380,717381,717382,717383,717385,717387,717388,717389,717390,717391,717392,717394,717395,717396,717397,717398,717399,717400,717401,717403,717404,717406,717407,717408,717411,717412,717413,717414,717415,717416,717417,717418,717422,717423,717424,717425,717427,717428,717430,717432,717433,717434,717435,717436,717438,717439,717440,717441,717442,717443,717444,717445,717446,717447,717448,717449,717450,717452,717453,717454,717455,717456,717459,717460,717461,717462,717463,717467,717468,717469,717470,717471,717472,717473,717475,717476,717477,717478,717479,717480,717481,717482,717484,717485,717486,717487,717488,717489,717490,717491,717492,717493,717495,717496,717497,717498,717500,717501,717502,717504,717505,717507,717508,717509,717510,717511,717513,717514,717515,717517,717518,717519,717520,717521,717522,717523,717525,717526,717527,717528,717529,717530,717531,717532,717534,717535,717536,717538,717539,717541,717542,717543,717545,717546,717547,717548,717549,717550,717551,717552,717553,717554,717556,717557,717558,717559,717560,717561,717562,717563,717564,717565,717566,717567,717568,717569,717570,717571,717572,717573,717574,717575,717576,717577,717578,717579,717581,717582,717583,717584,717585,717586,717587,717588,717589,717590,717592,717593,717595,717596,717597,717598,717599,717600,717601,717603,717604,717605,717606,717607,717608,717609,717610,717612,717613,717614,717615,717616,717617,717618,717619,717620,717621,717622,717624,717625,717626,717628,717629,717630,717631,717633,717634,717635,717636,717637,717638,717639,717640,717641,717642,717644,717645,717646,717647,717648,717649,717650,717651,717652,717653,717655,717656,717657,717658,717659,717660,717661,717662,717663,717664,717665,717666,717667,717668,717669,717670,717671,717672,717673,717674,717675,717676,717678,717679,717680,717681,717682,717683,717684,717685,717686,717687,717688,717689,717690,717691,717692,717693,717694,717695,717696,717697,717698,717700,717701,717702,717703,717704,717705,717706,717710,717712,717713,717714,717715,717717,717718,717719,717720,717721,717723,717724,717725,717727,717729,717730,717731,717732,717733,717734,717735,717736,717737,717738,717739,717740,717741,717742,717743,717744,717745,717746,717747,717748,717749,717751,717753,717755,717756,717757,717758,717759,717760,717761,717762,717763,717764,717765,717766,717767,717768,717770,717771,717772,717773,717774,717775,717776,717777,717779,717780,717781,717782,717783,717784,717785,717787,717788,717789,717790,717791,717792,717793,717794,717796,717798,717799,717800,717801,717802,717803,717804,717805,717807,717808,717809,717810,717811,717812,717814,717816,717819,717821,717823,717824,717825,717826,717827,717828,717829,717830,717831,717832,717833,717834,717835,717836,717837,717838,717839,717841,717842,717843,717844,717845,717846,717847,717848,717849,717850,717852,717853,717854,717855,717856,717857,717858,717859,717860,717861,717862,717864,717865,717866,717867,717868,717869,717870,717872,717878,717879,717880,717881,717882,717883,717884,717885,717886,717887,717890,717891,717892,717893,717894,717895,717896,717897,717898,717899,717900,717901,717903,717904,717905,717906,717907,717908,717909,717910,717911,717912,717913,717914,717915,717916,717917,717918,717919,717920,717921,717922,717923,717924,717925,717927,717928,717929,717930,717931,717932,717933,717935,717936,717937,717939,717940,717941,717942,717943,717944,717946,717947,717948,717949,717950,717951,717952,717953,717954,717956,717957,717958,717961,717962,717964,717965,717967,717968,717969,717970,717972,717973,717974,717975,717976,717977,717978,717979,717980,717981,717982,717984,717985,717986,717987,717989,717991,717992,717993,717994,717995,717996,717997,717998,717999,718000,718001,718003,718004,718006,718007,718008,718009,718010,718011,718013,718016,718017,718018,718019,718020,718021,718022,718023,718024,718027,718028,718029,718030,718032,718033,718035,718037,718038,718039,718040,718043,718044,718045,718046,718047,718048,718049,718050,718051,718052,718053,718054,718055,718057,718058,718059,718060,718061,718062,718063,718064,718065,718066,718069,718070,718071,718072,718073,718076,718077,718078,718079,718080,718081,718082,718083,718084,718085,718086,718087,718088,718089,718090,718092,718094,718095,718096,718097,718098,718099,718100,718101,718102,718103,718104,718105,718106,718107,718108,718109,718110,718111,718112,718113,718114,718115,718116,718117,718118,718119,718120,718121,718122,718123,718126,718127,718129,718130,718131,718132,718133,718134,718135,718136,718137,718138,718139,718140,718141,718142,718143,718144,718145,718146,718147,718148,718149,718150,718151,718152,718153,718155,718156,718158,718161,718163,718164,718166,718167,718168,718169,718170,718172,718173,718174,718175,718176,718177,718179,718180,718181,718182,718183,718184,718185,718186,718188,718189,718190,718191,718192,718193,718194,718195,718196,718197,718198,718199,718200,718201,718205,718206,718207,718209,718210,718212,718213,718214,718215,718216,718217,718218,718219,718220,718222,718223,718224,718225,718226,718227,718228,718229,718230,718231,718232,718233,718234,718235,718236,718237,718238,718239,718240,718241,718242,718243,718245,718247,718248,718249,718250,718251,718253,718254,718258,718260,718261,718262,718263,718265,718266,718267,718268,718269,718271,718272,718273,718274,718276,718277,718278,718280,718281,718282,718283,718284,718286,718287,718288,718289,718290,718292,718293,718294,718295,718296,718297,718299,718300,718301,718302,718304,718305,718307,718308,718309,718310,718311,718312,718313,718316,718319,718320,718321,718324,718326,718327,718328,718329,718330,718331,718332,718333,718334,718335,718336,718338,718339,718340,718341,718342,718343,718344,718345,718346,718347,718348,718349,718350,718352,718354,718356,718357,718360,718361,718362,718363,718364,718365,718366,718367,718368,718370,718371,718372,718373,718374,718375,718376,718377,718378,718379,718380,718384,718385,718386,718387,718388,718389,718390,718391,718392,718393,718394,718395,718396,718397,718398,718399,718400,718401,718402,718403,718404,718405,718407,718408,718409,718410,718413,718414,718415,718416,718417,718418,718420,718421,718423,718425,718427,718428,718429,718430,718432,718433,718434,718437,718438,718439,718441,718442,718443,718444,718445,718446,718447,718448,718449,718450,718451,718452,718455,718456,718457,718458,718459,718460,718461,718462,718463,718464,718465,718466,718467,718468,718469,718470,718471,718473,718474,718475,718476,718477,718478,718479,718481,718482,718483,718486,718487,718489,718490,718491,718492,718493,718494,718495,718496,718497,718498,718499,718500,718502,718504,718505,718508,718509,718510,718511,718513,718514,718515,718516,718517,718518,718519,718521,718522,718523,718524,718525,718526,718528,718531,718533,718534,718535,718536,718537,718538,718539,718540,718541,718542,718544,718547,718549,718550,718551,718552,718553,718555,718556,718557,718558,718560,718561,718563,718564,718565,718566,718567,718568,718569,718570,718571,718572,718573,718574,718575,718576,718581,718584,718585,718586,718589,718591,718592,718593,718596,718597,718598,718599,718601,718602,718603,718604,718606,718609,718610,718611,718613,718614,718615,718617,718619,718620,718621,718622,718623,718624,718625,718626,718627,718628,718630,718631,718632,718634,718635,718637,718638,718639,718641,718642,718644,718645,718646,718647,718648,718649,718651,718653,718654,718657,718658,718659,718661,718663,718664,718665,718666,718667,718668,718671,718672,718673,718674,718677,718678,718679,718680,718681,718682,718683,718684,718686,718687,718689,718690,718691,718692,718693,718694,718696,718697,718699,718700,718701,718702,718703,718704,718705,718706,718707,718708,718711,718712,718713,718714,718715,718716,718717,718721,718722,718723,718724,718725,718726,718727,718728,718729,718730,718731,718732,718734,718736,718738,718739,718740,718741,718742,718743,718744,718745,718746,718748,718749,718751,718752,718753,718754,718755,718756,718757,718758,718759,718762,718764,718765,718766,718767,718768,718769,718770,718772,718774,718775,718776,718778,718780,718781,718782,718783,718784,718785,718786,718788,718791,718792,718795,718796,718797,718799,718800,718801,718802,718803,718804,718805,718806,718807,718808,718809,718811,718813,718816,718817,718818,718820,718821,718822,718825,718826,718827,718828,718829,718831,718832,718833,718834,718835,718836,718837,718838,718840,718841,718842,718843,718844,718845,718846,718847,718848,718849,718850,718851,718852,718853,718854,718855,718856,718857,718858,718860,718861,718863,718864,718865,718866,718867,718869,718871,718872,718873,718874,718875,718876,718877,718878,718879,718880,718881,718882,718883,718884,718885,718886,718887,718889,718890,718891,718892,718893,718894,718895,718897,718898,718899,718901,718902,718903,718904,718905,718906,718907,718908,718909,718910,718911,718912,718913,718915,718918,718919,718921,718922,718923,718924,718925,718927,718928,718929,718931,718933,718936,718937,718938,718939,718941,718943,718947,718948,718950,718951,718952,718953,718954,718955,718956,718958,718959,718960,718961,718962,718964,718965,718966,718968,718969,718970,718973,718976,718978,718979,718980,718981,718982,718983,718984,718985,718987,718988,718990,718992,718993,718995,719000,719001,719002,719003,719004,719005,719006,719007,719008,719010,719011,719012,719013,719014,719015,719016,719017,719018,719019,719020,719021,719022,719024,719025,719026,719027,719029,719032,719033,719034,719035,719036,719037,719039,719041,719042,719043,719045,719046,719047,719048,719049,719051,719053,719054,719056,719057,719058,719059,719060,719061,719065,719066,719067,719069,719070,719071,719073,719074,719075,719076,719077,719078,719079,719080,719081,719082,719083,719084,719085,719087,719089,719090,719091,719092,719094,719095,719096,719097,719098,719099,719100,719101,719102,719103,719104,719105,719107,719108,719109,719110,719111,719112,719113,719114,719115,719117,719118,719119,719120,719121,719122,719124,719125,719126,719128,719130,719131,719132,719133,719135,719136,719139,719140,719141,719142,719143,719144,719145,719146,719147,719148,719149,719150,719151,719152,719153,719154,719157,719158,719159,719160,719161,719162,719163,719165,719166,719168,719169,719170,719171,719172,719173,719174,719175,719176,719177,719178,719180,719181,719182,719183,719186,719187,719188,719189,719190,719191,719193,719194,719195,719196,719197,719198,719199,719201,719202,719204,719205,719206,719207,719208,719209,719211,719212,719214,719215,719216,719218,719219,719220,719221,719222,719223,719227,719228,719230,719231,719232,719234,719235,719236,719237,719238,719239,719240,719241,719242,719243,719244,719245,719247,719248,719249,719250,719251,719252,719253,719254,719256,719257,719259,719261,719262,719263,719264,719265,719266,719267,719268,719269,719270,719271,719273,719275,719276,719278,719279,719280,719281,719282,719283,719284,719285,719287,719288,719289,719290,719291,719292,719294,719296,719297,719298,719300,719301,719302,719303,719305,719306,719307,719308,719309,719311,719312,719314,719315,719317,719318,719319,719320,719321,719322,719323,719324,719327,719328,719329,719330,719331,719332,719333,719334,719335,719336,719337,719338,719339,719340,719341,719342,719343,719344,719345,719346,719347,719348,719349,719350,719352,719353,719354,719355,719356,719357,719359,719360,719361,719362,719363,719364,719365,719366,719368,719370,719371,719372,719373,719375,719376,719377,719379,719380,719381,719383,719385,719388,719389,719390,719391,719393,719394,719396,719399,719400,719401,719402,719403,719404,719406,719409,719410,719411,719412,719413,719414,719415,719416,719417,719418,719420,719421,719422,719424,719426,719427,719428,719429,719430,719431,719432,719433,719434,719435,719436,719437,719440,719442,719443,719444,719445,719446,719447,719448,719449,719450,719451,719452,719453,719454,719455,719456,719457,719459,719461,719463,719464,719465,719466,719468,719469,719470,719471,719472,719474,719475,719476,719477,719479,719481,719482,719483,719484,719485,719486,719487,719488,719489,719490,719491,719492,719494,719495,719496,719497,719498,719499,719500,719501,719503,719504,719506,719507,719509,719510,719512,719513,719514,719515,719516,719517,719518,719519,719520,719521,719523,719524,719525,719526,719527,719528,719529,719533,719534,719535,719536,719538,719540,719541,719542,719544,719545,719547,719548,719550,719551,719552,719553,719554,719555,719556,719557,719558,719559,719560,719561,719563,719564,719565,719567,719568,719569,719570,719571,719572,719573,719574,719575,719577,719579,719580,719581,719582,719584,719585,719586,719587,719589,719590,719591,719592,719593,719594,719595,719596,719598,719599,719600,719601,719604,719605,719606,719607,719608,719609,719610,719611,719612,719613,719614,719616,719618,719619,719620,719621,719622,719623,719624,719626,719627,719629,719630,719631,719632,719633,719634,719635,719637,719639,719640,719641,719642,719643,719644,719645,719646,719647,719648,719649,719650,719651,719652,719653,719654,719655,719656,719657,719658,719660,719661,719662,719664,719665,719666,719669,719670,719671,719672,719673,719674,719676,719677,719678,719679,719680,719681,719682,719683,719684,719685,719686,719687,719688,719689,719690,719691,719692,719693,719694,719698,719699,719700,719701,719702,719703,719705,719706,719707,719708,719709,719710,719711,719712,719713,719714,719715,719717,719719,719720,719721,719723,719724,719725,719727,719728,719729,719730,719731,719732,719733,719735,719736,719737,719738,719740,719741,719742,719743,719744,719745,719746,719747,719748,719749,719750,719751,719752,719753,719754,719756,719757,719758,719759,719760,719761,719762,719763,719766,719767,719769,719770,719771,719772,719773,719774,719775,719776,719777,719778,719779,719780,719781,719782,719783,719785,719788,719789,719790,719791,719793,719795,719796,719797,719798,719799,719800,719801,719802,719803,719804,719805,719806,719807,719808,719809,719810,719811,719812,719813,719814,719815,719816,719817,719818,719819,719820,719821,719822,719823,719824,719825,719826,719827,719829,719830,719831,719832,719833,719834,719835,719836,719837,719839,719840,719841,719843,719844,719845,719846,719847,719848,719849,719850,719851,719852,719853,719856,719857,719858,719859,719860,719861,719862,719863,719864,719865,719866,719868,719869,719872,719873,719874,719875,719877,719879,719880,719881,719882,719883,719884,719885,719886,719887,719889,719891,719892,719893,719894,719895,719896,719897,719898,719899,719900,719901,719902,719904,719905,719906,719907,719908,719909,719910,719911,719912,719913,719914,719915,719916,719919,719920,719921,719922,719923,719924,719926,719927,719928,719929,719930,719932,719933,719934,719936,719937,719939,719940,719941,719942,719945,719946,719947,719948,719950,719951,719952,719954,719956,719958,719959,719960,719962,719964,719965,719966,719967,719968,719969,719970,719971,719972,719973,719974,719975,719976,719977,719978,719979,719980,719981,719982,719983,719984,719986,719988,719990,719992,719993,719994,719995,719996,719997,719998,719999,720002,720003,720004,720005,720006,720007,720008,720009,720010,720011,720012,720013,720014,720015,720016,720018,720019,720020,720021,720023,720024,720025,720027,720028,720030,720031,720032,720033,720034,720035,720036,720037,720038,720039,720041,720042,720043,720044,720045,720046,720047,720048,720049,720050,720051,720052,720053,720054,720055,720056,720057,720058,720059,720060,720062,720065,720066,720069,720071,720072,720074,720075,720076,720078,720079,720080,720081,720082,720083,720084,720085,720086,720087,720088,720089,720090,720091,720092,720094,720095,720096,720097,720098,720099,720100,720101,720102,720103,720104,720105,720106,720107,720108,720110,720113,720115,720117,720118,720119,720120,720123,720124,720125,720126,720127,720129,720130,720131,720132,720133,720134,720135,720136,720137,720139,720140,720141,720142,720143,720144,720145,720146,720147,720148,720151,720152,720154,720155,720156,720157,720158,720159,720160,720161,720162,720163,720164,720165,720166,720167,720168,720169,720170,720173,720174,720176,720177,720178,720179,720180,720182,720183,720184,720185,720186,720187,720189,720191,720192,720194,720195,720196,720198,720199,720201,720202,720203,720204,720205,720206,720207,720208,720209,720211,720212,720213,720214,720215,720217,720218,720220,720222,720223,720224,720225,720226,720227,720230,720231,720233,720235,720236,720237,720238,720239,720240,720241,720242,720243,720244,720246,720247,720248,720249,720250,720252,720253,720254,720255,720256,720258,720259,720260,720261,720262,720264,720266,720268,720269,720270,720271,720272,720274,720275,720276,720277,720279,720280,720283,720284,720285,720286,720287,720288,720290,720291,720292,720293,720294,720295,720297,720298,720299,720301,720302,720303,720304,720306,720307,720308,720309,720310,720311,720312,720313,720314,720315,720316,720317,720318,720320,720321,720322,720323,720324,720325,720326,720327,720328,720329,720330,720331,720332,720333,720334,720336,720337,720339,720340,720341,720343,720344,720345,720347,720348,720349,720350,720351,720352,720353,720354,720355,720356,720357,720358,720359,720360,720361,720362,720363,720364,720365,720367,720368,720369,720370,720371,720373,720374,720375,720376,720377,720378,720379,720380,720381,720382,720383,720384,720386,720387,720388,720389,720390,720391,720392,720393,720394,720395,720396,720397,720398,720399,720400,720401,720403,720404,720405,720406,720408,720409,720410,720411,720412,720413,720414,720415,720416,720417,729429,729430,729431,729432,729433,729434,729435,729436,729437,729438,729439,729440,729441,729442,729443,729444,729445,729446,729447,729448,729449,729450,729451,729452,729453,729454,729455,729456,729457,729458,729459,729460,729461,729462,729463,729464,729465,729466,729467,729468,729469,729470,729471,729472,729473,729474,729475,729476,729477,729478,729479,729480,729481,729482,729483,729484,729485,729486,729487,729488,729489,729490,729491,729492,729493,729494,729495,729496,729497,729498,729499,729500,729501,729502,729503,729504,729505,729506,729507,729508,729509,729510,729511,729512,729513,729514,729515,729516,729517,729518,729519,729520,729521,729522,729523,729524,729525,729526,729527,729528,729529,729530,729531,729532,729533,729534,729535,729536,729537,729538,729539,729540,729541,729542,729543,729544,729545,729546,729547,729548,729549,729550,729551,729552,729553,729554,729555,729556,729557,729558,729559,729560,729561,729562,729563,729564,729565,729566,729567,729568,729569,729570,729571,729572,729573,729574,729575,729576,729577,729578,729579,729580,729581,729582,729583,729584,729585,729586,729587,729588,729589,729590,729591,729592,729593,729594,729595,729596,729597,729598,729599,729600,729601,729602,729603,729604,729605,729606,729607,729608,729609,729610,729611,729612,729613,729614,729615,729616,729617,729618,729619,729620,729621,729622,729623,729624,729625,729626,729627,729628,729629,729630,729631,729632,729633,729634,729635,729636,729637,729638,729639,729640,729641,729642,729643,729644,729645,729646,729647,729648,729649,729650,729651,729652,729653,729654,729655,729656,729657,729658,729659,729660,729661,729662,729663,729664,729665,729666,729667,729668,729669,729670,729671,729672,729673,729674,729675,729676,729677,729678,729679,729680,729681,729682,729683,729684,729685,729686,729687,729688,729689,729690,729691,729692,729693,729694,729695,729696,729697,729698,729699,729700,729701,729702,729703,729704,729705,729706,729707,729708,729709,729710,729711,729712,729713,729714,729715,729716,729717,729718,729719,729720,729721,729722,729723,729724,729725,729726,729727,729728,729729,729730,729731,729732,729733,729734,729735,729736,729737,729738,729739,729740,729741,729742,729743,729744,729745,729746,729747,729748,729749,729750,729751,729752,729753,729754,729755,729756,729757,729758,729759,729760,729761,729762,729763,729764,729765,729766,729767,729768,729769,729770,729771,729772,729773,729774,729775,729776,729777,729778,729779,729780,729781,729782,729783,729784,729785,729786,729787,729788,729789,729790,729791,729792,729793,729794,729795,729796,729797,729798,729799,729800,729801,729802,729803,729804,729805,729806,729807,729808,729809,729810,729811,729812,729813,729814,729815,729816,729817,729818,729819,729820,729821,729822,729823,729824,729825,729826,729827,729828,729829,729830,729831,729832,729833,729834,729835,729836,729837,729838,729839,729840,729841,729842,729843,729844,729845,729846,729847,729848,729849,729850,729851,729852,729853,729854,729855,729856,729857,729858,729859,729860,729861,729862,729863,729864,729865,729866,729867,729868,729869,729870,729871,729872,729873,729874,729875,729876,729877,729878,729879,729880,729881,729882,729883,729884,729885,729886,729887,729888,729889,729890,729891,729892,729893,729894,729895,729896,729897,729898,729899,729900,729901,729902,729903,729904,729905,729906,729907,729908,729909,729910,729911,729912,729913,729914,729915,729916,729917,729918,729919,729920,729921,729922,729923,729924,729925,729926,729927,729928,729929,729930,729931,729932,729933,729934,729935,729936,729937,729938,729939,729940,729941,729942,729943,729944,729945,729946,729947,729948,729949,729950,729951,729952,729953,729954,729955,729956,729957,729958,729959,729960,729961,729962,729963,729964,729965,729966,729967,729968,729969,729970,729971,729972,729973,729974,729975,729976,729977,729978,729979,729980,729981,729982,729983,729984,729985,729986,729987,729988,729989,729990,729991,729992,729993,729994,729995,729996,729997,729998,729999,730000,730001,730002,730003,730004,730005,730006,730007,730008,730009,730010,730011,730012,730013,730014,730015,730016,730017,730018,730019,730020,730021,730022,730023,730024,730025,730026,730027,730028,730029,730030,730031,730032,730033,730034,730035,730036,730037,730038,730039,730040,730041,730042,730043,730044,730045,730046,730047,730048,730049,730050,730051,730052,730053,730054,730055,730056,730057,730058,730059,730060,730061,730062,730063,730064,730065,730066,730067,730068,730069,730070,730071,730072,730073,730074,730075,730076,730077,730078,730079,730080,730081,730082,730083,730084,730085,730086,730087,730088,730089,730090,730091,730092,730093,730094,730095,730096,730097,730098,730099,730100,730101,730102,730103,730104,730105,730106,730107,730108,730109,730110,730111,730112,730113,730114,730115,730116,730117,730118,730119,730120,730121,730122,730123,730124,730125,730126,730127,730128,730129,730130,730131,730132,730133,730134,730135,730136,730137,730138,730139,730140,730141,730142,730143,730144,730145,730146,730147,730148,730149,730150,730151,730152,730153,730154,730155,730156,730157,730158,730159,730160,730161,730162,730163,730164,730165,730166,730167,730168,730169,730170,730171,730172,730173,730174,730175,730176,730177,730178,730179,730180,730181,730182,730183,730184,730185,730186,730187,730188,730189,730190,730191,730192,730193,730194,730195,730196,730197,730198,730199,730200,730201,730202,730203,730204,730205,730206,730207,730208,730209,730210,730211,730212,730213,730214,730215,730216,730217,730218,730219,730220,730221,730222,730223,730224,730225,730226,730227,730228,730229,730230,730231,730232,730233,730234,730235,730236,730237,730238,730239,730240,730241,730242,730243,730244,730245,730246,730247,730248,730249,730250,730251,730252,730253,730254,730255,730256,730257,730258,730259,730260,730261,730262,730263,730264,730265,730266,730267,730268,730269,730270,730271,730272,730273,730274,730275,730276,730277,730278,730279,730280,730281,730282,730283,730284,730285,730286,730287,730288,730289,730290,730291,730292,730293,730294,730295,730296,730297,730298,730299,730300,730301,730302,730303,730304,730305,730306,730307,730308,730309,730310,730311,730312,730313,730314,730315,730316,730317,730318,730319,730320,730321,730322,730323,730324,730325,730326,730327,730328,730329,730330,730331,730332,730333,730334,730335,730336,730337,730338,730339,730340,730341,730342,730343,730344,730345,730346,730347,730348,730349,730350,730351,730352,730353,730354,730355,730356,730357,730358,730359,730360,730361,730362,730363,730364,730365,730366,730367,730368,730369,730370,730371,730372,730373,730374,730375,730376,730377,730378,730379,730380,730381,730382,730383,730384,730385,730386,730387,730388,730389,730390,730391,730392,730393,730394,730395,730396,730397,730398,730399,730400,730401,730402,730403,730404,730405,730406,730407,730408,730409,730410,730411,730412,730413,730414,730415,730416,730417,730418,730419,730420,730421,730422,730423,730424,730425,730426,730427,730428,730429,730430,730431,730432,730433,730434,730435,730436,730437,730438,730439,730440,730441,730442,730443,730444,730445,730446,730447,730448,730449,730450,730451,730452,730453,730454,730455,730456,730457,730458,730459,730460,730461,730462,730463,730464,730465,730466,730467,730468,730469,730470,730471,730472,730473,730474,730475,730476,730477,730478,730479,730480,730481,730482,730483,730484,730485,730486,730487,730488,730489,730490,730491,730492,730493,730494,730495,730496,730497,730498,730499,730500,730501,730502,730503,730504,730505,730506,730507,730508,730509,730510,730511,730512,730513,730514,730515,730516,730517,730518,730519,730520,730521,730522,730523,730524,730525,730526,730527,730528,730529,730530,730531,730532,730533,730534,730535,730536,730537,730538,730539,730540,730541,730542,730543,730544,730545,730546,730547,730548,730549,730550,730551,730552,730553,730554,730555,730556,730557,730558,730559,730560,730561,730562,730563,730564,730565,730566,730567,730568,730569,730570,730571,730572,730573,730574,730575,730576,730577,730578,730579,730580,730581,730582,730583,730584,730585,730586,730587,730588,730589,730590,730591,730592,730593,730594,730595,730596,730597,730598,730599,730600,730601,730602,730603,730604,730605,730606,730607,730608,730609,730610,730611,730612,730613,730614,730615,730616,730617,730618,730619,730620,730621,730622,730623,730624,730625,730626,730627,730628,730629,730630,730631,730632,730633,730634,730635,730636,730637,730638,730639,730640,730641,730642,730643,730644,730645,730646,730647,730648,730649,730650,730651,730652,730653,730654,730655,730656,730657,730658,730659,730660,730661,730662,730663,730664,730665,730666,730667,730668,730669,730670,730671,730672,730673,730674,730675,730676,730677,730678,730679,730680,730681,730682,730683,730684,730685,730686,730687,730688,730689,730690,730691,730692,730693,730694,730695,730696,730697,730698,730699,730700,730701,730702,730703,730704,730705,730706,730707,730708,730709,730710,730711,730712,730713,730714,730715,730716,730717,730718,730719,730720,730721,730722,730723,730724,730725,730726,730727,730728,730729,730730,730731,730732,730733,730734,730735,730736,730737,730738,730739,730740,730741,730742,730743,730744,730745,730746,730747,730748,730749,730750,730751,730752,730753,730754,730755,730756,730757,730758,730759,730760,730761,730762,730763,730764,730765,730766,730767,730768,730769,730770,730771,730772,730773,730774,730775,730776,730777,730778,730779,730780,730781,730782,730783,730784,730785,730786,730787,730788,730789,730790,730791,730792,730793,730794,730795,730796,730797,730798,730799,730800,730801,730802,730803,730804,730805,730806,730807,730808,730809,730810,730811,730812,730813,730814,730815,730816,730817,730818,730819,730820,730821,730822,730823,730824,730825,730826,730827,730828,730829,730830,730831,730832,730833,730834,730835,730836,730837,730838,730839,730840,730841,730842,730843,730844,730845,730846,730847,730848,730849,730850,730851,730852,730853,730854,730855,730856,730857,730858,730859,730860,730861,730862,730863,730864,730865,730866,730867,730868,730869,730870,730871,730872,730873,730874,730875,730876,730877,730878,730879,730880,730881,730882,730883,730884,730885,730886,730887,730888,730889,730890,730891,730892,730893,730894,730895,730896,730897,730898,730899,730900,730901,730902,730903,730904,730905,730906,730907,730908,730909,730910,730911,730912,730913,730914,730915,730916,730917,730918,730919,730920,730921,730922,730923,730924,730925,730926,730927,730928,730929,730930,730931,730932,730933,730934,730935,730936,730937,730938,730939,730940,730941,730942,730943,730944,730945,730946,730947,730948,730949,730950,730951,730952,730953,730954,730955,730956,730957,730958,730959,730960,730961,730962,730963,730964,730965,730966,730967,730968,730969,730970,730971,730972,730973,730974,730975,730976,730977,730978,730979,730980,730981,730982,730983,730984,730985,730986,730987,730988,730989,730990,730991,730992,730993,730994,730995,730996,730997,730998,730999,731000,731001,731002,731003,731004,731005,731006,731007,731008,731009,731010,731011,731012,731013,731014,731015,731016,731017,731018,731019,731020,731021,731022,731023,731024,731025,731026,731027,731028,731029,731030,731031,731032,731033,731034,731035,731036,731037,731038,731039,731040,731041,731042,731043,731044,731045,731046,731047,731048,731049,731050,731051,731052,731053,731054,731055,731056,731057,731058,731059,731060,731061,731062,731063,731064,731065,731066,731067,731068,731069,731070,731071,731072,731073,731074,731075,731076,731077,731078,731079,731080,731081,731082,731083,731084,731085,731086,731087,731088,731089,731090,731091,731092,731093,731094,731095,731096,731097,731098,731099,731100,731101,731102,731103,731104,731105,731106,731107,731108,731109,731110,731111,731112,731113,731114,731115,731116,731117,731118,731119,731120,731121,731122,731123,731124,731125,731126,731127,731128,731129,731130,731131,731132,731133,731134,731135,731136,731137,731138,731139,731140,731141,731142,731143,731144,731145,731146,731147,731148,731149,731150,731151,731152,731153,731154,731155,731156,731157,731158,731159,731160,731161,731162,731163,731164,731165,731166,731167,731168,731169,731170,731171,731172,731173,731174,731175,731176,731177,731178,731179,731180,731181,731182,731183,731184,731185,731186,731187,731188,731189,731190,731191,731192,731193,731194,731195,731196,731197,731198,731199,731200,731201,731202,731203,731204,731205,731206,731207,731208,731209,731210,731211,731212,731213,731214,731215,731216,731217,731218,731219,731220,731221,731222,731223,731224,731225,731226,731227,731228,731229,731230,731231,731232,731233,731234,731235,731236,731237,731238,731239,731240,731241,731242,731243,731244,731245,731246,731247,731248,731249,731250,731251,731252,731253,731254,731255,731256,731257,731258,731259,731260,731261,731262,731263,731264,731265,731266,731267,731268,731269,731270,731271,731272,731273,731274,731275,731276,731277,731278,731279,731280,731281,731282,731283,731284,731285,731286,731287,731288,731289,731290,731291,731292,731293,731294,731295,731296,731297,731298,731299,731300,731301,731302,731303,731304,731305,731306,731307,731308,731309,731310,731311,731312,731313,731314,731315,731316,731317,731318,731319,731320,731321,731322,731323,731324,731325,731326,731327,731328,731329,731330,731331,731332,731333,731334,731335,731336,731337,731338,731339,731340,731341,731342,731343,731344,731345,731346,731347,731348,731349,731350,731351,731352,731353,731354,731355,731356,731357,731358,731359,731360,731361,731362,731363,731364,731365,731366,731367,731368,731369,731370,731371,731372,731373,731374,731375,731376,731377,731378,731379,731380,731381,731382,731383,731384,731385,731386,731387,731388,731389,731390,731391,731392,731393,731394,731395,731396,731397,731398,731399,731400,731401,731402,731403,731404,731405,731406,731407,731408,731409,731410,731411,731412,731413,731414,731415,731416,731417,731418,731419,731420,731421,731422,731423,731424,731425,731426,731427,731428,731429,731430,731431,731432,731433,731434,731435,731436,731437,731438,731439,731440,731441,731442,731443,731444,731445,731446,731447,731448,731449,731450,731451,731452,731453,731454,731455,731456,731457,731458,731459,731460,731461,731462,731463,731464,731465,731466,731467,731468,731469,731470,731471,731472,731473,731474,731475,731476,731477,731478,731479,731480,731481,731482,731483,731484,731485,731486,731487,731488,731489,731490,731491,731492,731493,731494,731495,731496,731497,731498,731499,731500,731501,731502,731503,731504,731505,731506,731507,731508,731509,731510,731511,731512,731513,731514,731515,731516,731517,731518,731519,731520,731521,731522,731523,731524,731525,731526,731527,731528,731529,731530,731531,731532,731533,731534,731535,731536,731537,731538,731539,731540,731541,731542,731543,731544,731545,731546,731547,731548,731549,731550,731551,731552,731553,731554,731555,731556,731557,731558,731559,731560,731561,731562,731563,731564,731565,731566,731567,731568,731569,731570,731571,731572,731573,731574,731575,731576,731577,731578,731579,731580,731581,731582,731583,731584,731585,731586,731587,731588,731589,731590,731591,731592,731593,731594,731595,731596,731597,731598,731599,731600,731601,731602,731603,731604,731605,731606,731607,731608,731609,731610,731611,731612,731613,731614,731615,731616,731617,731618,731619,731620,731621,731622,731623,731624,731625,731626,731627,731628,731629,731630,731631,731632,731633,731634,731635,731636,731637,731638,731639,731640,731641,731642,731643,731644,731645,731646,731647,731648,731649,731650,731651,731652,731653,731654,731655,731656,731657,731658,731659,731660,731661,731662,731663,731664,731665,731666,731667,731668,731669,731670,731671,731672,731673,731674,731675,731676,731677,731678,731679,731680,731681,731682,731683,731684,731685,731686,731687,731688,731689,731690,731691,731692,731693,731694,731695,731696,731697,731698,731699,731700,731701,731702,731703,731704,731705,731706,731707,731708,731709,731710,731711,731712,731713,731714,731715,731716,731717,731718,731719,731720,731721,731722,731723,731724,731725,731726,731727,731728,731729,731730,731731,731732,731733,731734,731735,731736,731737,731738,731739,731740,731741,731742,731743,731744,731745,731746,731747,731748,731749,731750,731751,731752,731753,731754,731755,731756,731757,731758,731759,731760,731761,731762,731763,731764,731765,731766,731767,731768,731769,731770,731771,731772,731773,731774,731775,731776,731777,731778,731779,731780,731781,731782,731783,731784,731785,731786,731787,731788,731789,731790,731791,731792,731793,731794,731795,731796,731797,731798,731799,731800,731801,731802,731803,731804,731805,731806,731807,731808,731809,731810,731811,731812,731813,731814,731815,731816,731817,731818,731819,731820,731821,731822,731823,731824,731825,731826,731827,731828,731829,731830,731831,731832,731833,731834,731835,731836,731837,731838,731839,731840,731841,731842,731843,731844,731845,731846,731847,731848,731849,731850,731851,731852,731853,731854,731855,731856,731857,731858,731859,731860,731861,731862,731863,731864,731865,731866,731867,731868,731869,731870,731871,731872,731873,731874,731875,731876,731877,731878,731879,731880,731881,731882,731883,731884,731885,731886,731887,731888,731889,731890,731891,731892,731893,731894,731895,731896,731897,731898,731899,731900,731901,731902,731903,731904,731905,731906,731907,731908,731909,731910,731911,731912,731913,731914,731915,731916,731917,731918,731919,731920,731921,731922,731923,731924,731925,731926,731927,731928,731929,731930,731931,731932,731933,731934,731935,731936,731937,731938,731939,731940,731941,731942,731943,731944,731945,731946,731947,731948,731949,731950,731951,731952,731953,731954,731955,731956,731957,731958,731959,731960,731961,731962,731963,731964,731965,731966,731967,731968,731969,731970,731971,731972,731973,731974,731975,731976,731977,731978,731979,731980,731981,731982,731983,731984,731985,731986,731987,731988,731989,731990,731991,731992,731993,731994,731995,731996,731997,731998,731999,732000,732001,732002,732003,732004,732005,732006,732007,732008,732009,732010,732011,732012,732013,732014,732015,732016,732017,732018,732019,732020,732021,732022,732023,732024,732025,732026,732027,732028,732029,732030,732031,732032,732033,732034,732035,732036,732037,732038,732039,732040,732041,732042,732043,732044,732045,732046,732047,732048,732049,732050,732051,732052,732053,732054,732055,732056,732057,732058,732059,732060,732061,732062,732063,732064,732065,732066,732067,732068,732069,732070,732071,732072,732073,732074,732075,732076,732077,732078,732079,732080,732081,732082,732083,732084,732085,732086,732087,732088,732089,732090,732091,732092,732093,732094,732095,732096,732097,732098,732099,732100,732101,732102,732103,732104,732105,732106,732107,732108,732109,732110,732111,732112,732113,732114,732115,732116,732117,732118,732119,732120,732121,732122,732123,732124,732125,732126,732127,732128,732129,732130,732131,732132,732133,732134,732135,732136,732137,732138,732139,732140,732141,732142,732143,732144,732145,732146,732147,732148,732149,732150,732151,732152,732153,732154,732155,732156,732157,732158,732159,732160,732161,732162,732163,732164,732165,732166,732167,732168,732169,732170,732171,732172,732173,732174,732175,732176,732177,732178,732179,732180,732181,732182,732183,732184,732185,732186,732187,732188,732189,732190,732191,732192,732193,732194,732195,732196,732197,732198,732199,732200,732201,732202,732203,732204,732205,732206,732207,732208,732209,732210,732211,732212,732213,732214,732215,732216,732217,732218,732219,732220,732221,732222,732223,732224,732225,732226,732227,732228,732229,732230,732231,732232,732233,732234,732235,732236,732237,732238,732239,732240,732241,732242,732243,732244,732245,732246,732247,732248,732249,732250,732251,732252,732253,732254,732255,732256,732257,732258,732259,732260,732261,732262,732263,732264,732265,732266,732267,732268,732269,732270,732271,732272,732273,732274,732275,732276,732277,732278,732279,732280,732281,732282,732283,732284,732285,732286,732287,732288,732289,732290,732291,732292,732293,732294,732295,732296,732297,732298,732299,732300,732301,732302,732303,732304,732305,732306,732307,732308,732309,732310,732311,732312,732313,732314,732315,732316,732317,732318,732319,732320,732321,732322,732323,732324,732325,732326,732327,732328,732329,732330,732331,732332,732333,732334,732335,732336,732337,732338,732339,732340,732341,732342,732343,732344,732345,732346,732347,732348,732349,732350,732351,732352,732353,732354,732355,732356,732357,732358,732359,732360,732361,732362,732363,732364,732365,732366,732367,732368,732369,732370,732371,732372,732373,732374,732375,732376,732377,732378,732379,732380,732381,732382,732383,732384,732385,732386,732387,732388,732389,732390,732391,732392,732393,732394,732395,732396,732397,732398,732399,732400,732401,732402,732403,732404,732405,732406,732407,732408,732409,732410,732411,732412,732413,732414,732415,732416,732417,732418,732419,732420,732421,732422,732423,732424,732425,732426,732427,732428,732429,732430,732431,732432,732433,732434,732435,732436,732437,732438,732439,732440,732441,732442,732443,732444,732445,732446,732447,732448,732449,732450,732451,732452,732453,732454,732455,732456,732457,732458,732459,732460,732461,732462,732463,732464,732465,732466,732467,732468,732469,732470,732471,732472,732473,732474,732475,732476,732477,732478,732479,732480,732481,732482,732483,732484,732485,732486,732487,732488,732489,732490,732491,732492,732493,732494,732495,732496,732497,732498,732499,732500,732501,732502,732503,732504,732505,732506,732507,732508,732509,732510,732511,732512,732513,732514,732515,732516,732517,732518,732519,732520,732521,732522,732523,732524,732525,732526,732527,732528,732529,732530,732531,732532,732533,732534,732535,732536,732537,732538,732539,732540,732541,732542,732543,732544,732545,732546,732547,732548,732549,732550,732551,732552,732553,732554,732555,732556,732557,732558,732559,732560,732561,732562,732563,732564,732565,732566,732567,732568,732569,732570,732571,732572,732573,732574,732575,732576,732577,732578,732579,732580,732581,732582,732583,732584,732585,732586,732587,732588,732589,732590,732591,732592,732593,732594,732595,732596,732597,732598,732599,732600,732601,732602,732603,732604,732605,732606,732607,732608,732609,732610,732611,732612,732613,732614,732615,732616,732617,732618,732619,732620,732621,732622,732623,732624,732625,732626,732627,732628,732629,732630,732631,732632,732633,732634,732635,732636,732637,732638,732639,732640,732641,732642,732643,732644,732645,732646,732647,732648,732649,732650,732651,732652,732653,732654,732655,732656,732657,732658,732659,732660,732661,732662,732663,732664,732665,732666,732667,732668,732669,732670,732671,732672,732673,732674,732675,732676,732677,732678,732679,732680,732681,732682,732683,732684,732685,732686,732687,732688,732689,732690,732691,732692,732693,732694,732695,732696,732697,732698,732699,732700,732701,732702,732703,732704,732705,732706,732707,732708,732709,732710,732711,732712,732713,732714,732715,732716,732717,732718,732719,732720,732721,732722,732723,732724,732725,732726,732727,732728,732729,732730,732731,732732,732733,732734,732735,732736,732737,732738,732739,732740,732741,732742,732743,732744,732745,732746,732747,732748,732749,732750,732751,732752,732753,732754,732755,732756,732757,732758,732759,732760,732761,732762,732763,732764,732765,732766,732767,732768,732769,732770,732771,732772,732773,732774,732775,732776,732777,732778,732779,732780,732781,732782,732783,732784,732785,732786,732787,732788,732789,732790,732791,732792,732793,732794,732795,732796,732797,732798,732799,732800,732801,732802,732803,732804,732805,732806,732807,732808,732809,732810,732811,732812,732813,732814,732815,732816,732817,732818,732819,732820,732821,732822,732823,732824,732825,732826,732827,732828,732829,732830,732831,732832,732833,732834,732835,732836,732837,732838,732839,732840,732841,732842,732843,732844,732845,732846,732847,732848,732849,732850,732851,732852,732853,732854,732855,732856,732857,732858,732859,732860,732861,732862,732863,732864,732865,732866,732867,732868,732869,732870,732871,732872,732873,732874,732875,732876,732877,732878,732879,732880,732881,732882,732883,732884,732885,732886,732887,732888,732889,732890,732891,732892,732893,732894,732895,732896,732897,732898,732899,732900,732901,732902,732903,732904,732905,732906,732907,732908,732909,732910,732911,732912,732913,732914,732915,732916,732917,732918,732919,732920,732921,732922,732923,732924,732925,732926,732927,732928,732929,732930,732931,732932,732933,732934,732935,732936,732937,732938,732939,732940,732941,732942,732943,732944,732945,732946,732947,732948,732949,732950,732951,732952,732953,732954,732955,732956,732957,732958,732959,732960,732961,732962,732963,732964,732965,732966,732967,732968,732969,732970,732971,732972,732973,732974,732975,732976,732977,732978,732979,732980,732981,732982,732983,732984,732985,732986,732987,732988,732989,732990,732991,732992,732993,732994,732995,732996,732997,732998,732999,733000,733001,733002,733003,733004,733005,733006,733007,733008,733009,733010,733011,733012,733013,733014,733015,733016,733017,733018,733019,733020,733021,733022,733023,733024,733025,733026,733027,733028,733029,733030,733031,733032,733033,733034,733035,733036,733037,733038,733039,733040,733041,733042,733043,733044,733045,733046,733047,733048,733049,733050,733051,733052,733053,733054,733055,733056,733057,733058,733059,733060,733061,733062,733063,733064,733065,733066,733067,733068,733069,733070,733071,733072,733073,733074,733075,733076,733077,733078,733079,733080,733081,733082,733083,733084,733085,733086,733087,733088,733089,733090,733091,733092,733093,733094,733095,733096,733097,733098,733099,733100,733101,733102,733103,733104,733105,733106,733107,733108,733109,733110,733111,733112,733113,733114,733115,733116,733117,733118,733119,733120,733121,733122,733123]"))

LANES = ("direct", "wayback")
EXPECTED_LANGS = {"direct": {"vi"}, "wayback": {"vi"}}

DIRECT_MIN_DELAY_SECONDS = 1.0       # mỗi domain tối đa ~1 request/giây (robots.txt có Crawl-delay lớn hơn thì theo robots)
DIRECT_MAX_DELAY_SECONDS = 30.0
DIRECT_WORKERS_PER_DOMAIN = 6        # Kaggle → máy chủ VN mất vài giây/trang; cổng giãn cách vẫn giữ ≤1 request/giây/domain
BLOCK_SWITCH_AFTER = 15              # 15 lần liên tiếp bị chặn (403/401/trang kiểm tra bot) → domain chuyển sang Wayback

WAYBACK_HOST = "web.archive.org"
WAYBACK_TIMESTAMP = "2026"           # Wayback tự chuyển tới bản lưu gần nhất (mới nhất)
CDX_URL = "https://web.archive.org/cdx/search/cdx"
# Domain lớn có tiền tố chung: tra chỉ mục CDX một lần → mỗi bài chỉ 1 request (thay vì 2), URL không có bản lưu bỏ qua ngay.
CDX_PREFIXES = {"nhathuoclongchau.com.vn": "nhathuoclongchau.com.vn/bai-viet/", "youmed.vn": "youmed.vn/tin-tuc/"}
WAYBACK_MIN_DELAY_SECONDS = 2.0
WAYBACK_MAX_DELAY_SECONDS = 16.0
WAYBACK_WORKERS = 3
MAX_CONSECUTIVE_TRANSIENT_FAILURES = 30

MAX_RUNTIME_HOURS = 11.0
SHARD_DOCS = 10_000
MIN_TEXT_CHARS = 200
PROGRESS_SECONDS = 120
BUNDLE_EVERY_MINUTES = 60
MAX_FETCHES_PER_LANE = int(os.environ["CRAWL_LIMIT"]) if os.environ.get("CRAWL_LIMIT") else None

HTTP_TIMEOUT_SECONDS = 60
MAX_RESPONSE_BYTES = 5_000_000
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2.0

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or (os.name != "nt" and Path("/kaggle/working").is_dir())
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("CRAWL_INPUT_ROOT", "crawl_input"))
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("CRAWL_WORK_ROOT", "crawl_work"))
VIZ_ROOT = WORK_ROOT / "corpus" / "viz"
DOCS_DIR = VIZ_ROOT / "docs" / OWNER
LOGS_DIR = VIZ_ROOT / "crawl_log" / OWNER
MANIFESTS_DIR = VIZ_ROOT / "manifest" / OWNER
LINKS_LOCAL = WORK_ROOT / "input" / LINKS_FILENAME
STATE_PATH = WORK_ROOT / "state-a-prio.sqlite3"
BUNDLE_PATH = WORK_ROOT / "prio_resume.zip"
for _directory in (DOCS_DIR, LOGS_DIR, MANIFESTS_DIR, LINKS_LOCAL.parent):
    _directory.mkdir(parents=True, exist_ok=True)

print(f"Môi trường: {'Kaggle' if ON_KAGGLE else 'local'} | input={INPUT_ROOT} | working={WORK_ROOT}")
print(f"{len(TARGET_DOMAINS)} domain | trực tiếp ≥{DIRECT_MIN_DELAY_SECONDS:g} giây/request/domain | "
      f"Wayback {WAYBACK_MIN_DELAY_SECONDS:g}–{WAYBACK_MAX_DELAY_SECONDS:g} giây/request")
if MAX_FETCHES_PER_LANE is not None:
    print(f"Giới hạn thử nghiệm: tối đa {MAX_FETCHES_PER_LANE} URL mỗi luồng (mỗi domain trực tiếp / Wayback)")

In [ ]:
# Chuẩn hóa văn bản, phát hiện ngôn ngữ và kiểm tra schema — sao chép nguyên từ corpus_crawler (text.py, validation.py)
# để shard mới giống hệt shard đã có.

_CONTROL_OR_ZERO_WIDTH = re.compile(
    "["
    "\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f-\u009f"
    "­᠎​-‏‪-‮⁠-⁯﻿"
    "]"
)
_NEWLINES = re.compile(r"\n{3,}")
_SPACES = re.compile(r"[^\S\n]+")


def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFC", value).replace("\r\n", "\n").replace("\r", "\n")
    value = _CONTROL_OR_ZERO_WIDTH.sub("", value)
    value = _SPACES.sub(" ", value)
    value = _NEWLINES.sub("\n\n", value)
    return value.strip()


def normalize_hostname(url: str) -> str | None:
    try:
        hostname = urlsplit(url).hostname
    except ValueError:
        return None
    if not hostname:
        return None
    hostname = hostname.lower().rstrip(".")
    if hostname.startswith("www."):
        hostname = hostname[4:]
    return hostname or None


def detect_supported_language(text: str) -> tuple[str, float] | None:
    from langdetect import DetectorFactory, detect_langs
    from langdetect.lang_detect_exception import LangDetectException

    DetectorFactory.seed = 0
    try:
        candidates = detect_langs(text)
    except LangDetectException:
        return None
    if not candidates:
        return None
    detected = candidates[0]
    code = detected.lang.lower()
    if code.startswith("zh"):
        code = "zh"
    if code not in {"vi", "zh", "en"}:
        return None
    return code, float(detected.prob)


def timestamp_now() -> str:
    return datetime.now().astimezone().isoformat(timespec="seconds")


DOC_STATUSES = {"ok", "empty", "too_short", "http_error", "timeout", "blocked", "lang_mismatch", "duplicate"}
PASSAGE_TYPES = {"title", "abstract", "heading", "body"}
DOC_KEYS = (
    "schema_version", "uid", "doc_id", "lang", "source", "url", "domain", "title", "text",
    "norm_version", "passages", "text_len", "content_hash", "dup_of", "fetch", "pubmed",
    "owner", "code_commit",
)
FETCH_KEYS = ("status", "http_status", "fetched_at", "extractor", "lang_detected", "lang_conf")


def validate_doc(doc: dict, owner: str) -> None:
    if tuple(doc) != DOC_KEYS:
        raise ValueError(f"Thứ tự/tập key của document khác shard hiện có: {list(doc)}")
    if tuple(doc["fetch"]) != FETCH_KEYS:
        raise ValueError(f"Key trong fetch khác shard hiện có: {list(doc['fetch'])}")
    if doc["lang"] not in {"vi", "zh", "en"}:
        raise ValueError(f"Unsupported document language: {doc['lang']!r}")
    if not isinstance(doc["doc_id"], str) or not doc["doc_id"]:
        raise ValueError("doc_id must be a non-empty string")
    if doc["owner"] != owner:
        raise ValueError(f"Document owner must be {owner!r}")
    if doc["uid"] != f"{doc['lang']}:{doc['doc_id']}":
        raise ValueError("uid must be '{lang}:{doc_id}'")
    text = doc["text"]
    if not isinstance(text, str) or doc["text_len"] != len(text):
        raise ValueError("text_len must equal the Unicode character length of text")
    if not text:
        raise ValueError("Documents in a docs shard must have non-empty text")
    if doc["content_hash"] != "sha1:" + hashlib.sha1(text.encode("utf-8")).hexdigest():
        raise ValueError("content_hash does not match text")
    if doc["fetch"].get("status") not in DOC_STATUSES:
        raise ValueError("fetch.status is missing or invalid")
    for passage in doc["passages"]:
        if (
            passage.get("type") not in PASSAGE_TYPES
            or not isinstance(passage.get("start"), int)
            or not isinstance(passage.get("end"), int)
            or not 0 <= passage["start"] <= passage["end"] <= len(text)
        ):
            raise ValueError(f"Invalid passage offsets: {passage!r}")


def validate_shard_file(path: Path, owner: str) -> tuple[int, str]:
    seen: set[str] = set()
    count = 0
    with path.open("rb") as raw, zstandard.ZstdDecompressor().stream_reader(raw) as reader:
        buffer = b""
        while chunk := reader.read(1024 * 1024):
            buffer += chunk
            *lines, buffer = buffer.split(b"\n")
            for line in lines:
                if not line.strip():
                    continue
                doc = json.loads(line)
                validate_doc(doc, owner)
                if doc["uid"] in seen:
                    raise ValueError(f"Duplicate uid in shard: {doc['uid']}")
                seen.add(doc["uid"])
                count += 1
        if buffer.strip():
            raise ValueError(f"Dòng cuối của shard thiếu newline: {path}")
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return count, digest.hexdigest()


def build_records(
    *, source: str, source_id: int, url: str, domain: str, fetch_status: str,
    http_status: int | None, fetched_at: str, final_url: str | None, attempts: int,
    nbytes: int | None, error: str | None, raw_title: str | None, raw_body: str | None,
    doc_url: str, body_type: str, extractor: str, pubmed_meta: dict | None,
) -> tuple[dict, dict | None]:
    """Cùng logic với corpus_crawler.extract.make_records; chỉ khác ngôn ngữ kỳ vọng theo nguồn."""
    doc = None
    status = fetch_status
    if fetch_status == "ok":
        raw_title = raw_title or ""
        raw_body = raw_body or ""
        title = normalize_text(raw_title)
        article = normalize_text(raw_body)
        if not article:
            status = "empty"
        else:
            content = normalize_text(f"{raw_title}\n\n{raw_body}" if raw_title else raw_body)
            if len(content) < MIN_TEXT_CHARS:
                status = "too_short"
            else:
                detected = detect_supported_language(content)
                if detected is None:
                    status = "lang_mismatch"
                else:
                    language, confidence = detected
                    if language not in EXPECTED_LANGS[source]:
                        status = "lang_mismatch"
                    passages = []
                    if title and content.startswith(title):
                        passages.append({"type": "title", "start": 0, "end": len(title)})
                        body_start = content.find(article, len(title))
                    else:
                        body_start = content.find(article)
                    if body_start >= 0 and article:
                        passages.append({"type": body_type, "start": body_start, "end": body_start + len(article)})
                    doc = {
                        "schema_version": SCHEMA_VERSION,
                        "uid": f"{language}:{source_id}",
                        "doc_id": str(source_id),
                        "lang": language,
                        "source": "btc_url",
                        "url": doc_url,
                        "domain": domain,
                        "title": title,
                        "text": content,
                        "norm_version": NORM_VERSION,
                        "passages": passages,
                        "text_len": len(content),
                        "content_hash": "sha1:" + hashlib.sha1(content.encode("utf-8")).hexdigest(),
                        "dup_of": None,
                        "fetch": {
                            "status": status,
                            "http_status": http_status,
                            "fetched_at": fetched_at,
                            "extractor": extractor,
                            "lang_detected": language,
                            "lang_conf": confidence,
                        },
                        "pubmed": pubmed_meta,
                        "owner": OWNER,
                        "code_commit": CODE_COMMIT,
                    }
    log = {
        "uid": doc["uid"] if doc is not None else f"src:{source_id}",
        "id": source_id,
        "url": url,
        "domain": domain,
        "owner": OWNER,
        "status": status,
        "http_status": http_status,
        "attempt": attempts,
        "fetched_at": fetched_at,
        "bytes": nbytes,
        "final_url": final_url,
        "error": error,
        "shard": None,
    }
    if doc is not None:
        doc["fetch"]["status"] = status
    return log, doc

In [ ]:
# Checkpoint SQLite + khôi phục bundle + nạp URL từ file BTC + xếp ưu tiên theo truy vấn + ghi shard
# (cùng định dạng ShardWriter của corpus_crawler).

DOC_PREFIXES = tuple(f"viz-{OWNER}-{lane}-" for lane in LANES)
LOG_PREFIXES = tuple(f"log-{OWNER}-{lane}-" for lane in LANES)
LOG_SCHEMA = pa.schema([
    ("uid", pa.string()), ("id", pa.int64()), ("url", pa.string()), ("domain", pa.string()),
    ("owner", pa.string()), ("status", pa.string()), ("http_status", pa.int64()), ("attempt", pa.int64()),
    ("fetched_at", pa.string()), ("bytes", pa.int64()), ("final_url", pa.string()), ("error", pa.string()),
    ("shard", pa.string()),
])
MANIFEST_INPUT = {
    "direct": {"type": "domain_list", "value": f"{LINKS_FILENAME}#{len(TARGET_DOMAINS)} domain (trực tiếp, theo ưu tiên truy vấn)"},
    "wayback": {"type": "domain_list", "value": f"{LINKS_FILENAME}#{len(TARGET_DOMAINS)} domain via {WAYBACK_HOST} (id_ snapshot)"},
}
STATE_SCHEMA = """
CREATE TABLE IF NOT EXISTS urls (
    id INTEGER PRIMARY KEY,
    url TEXT NOT NULL,
    domain TEXT NOT NULL,
    source TEXT NOT NULL,
    priority INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'pending',
    result TEXT,
    attempt INTEGER NOT NULL DEFAULT 0,
    log_json TEXT,
    doc_json TEXT,
    doc_shard TEXT,
    doc_exported INTEGER NOT NULL DEFAULT 0,
    log_exported INTEGER NOT NULL DEFAULT 0
);
CREATE INDEX IF NOT EXISTS urls_claim ON urls(source, status, priority);
CREATE INDEX IF NOT EXISTS urls_domain ON urls(source, domain, status, priority);
CREATE INDEX IF NOT EXISTS urls_docs ON urls(source, doc_exported, id);
CREATE INDEX IF NOT EXISTS urls_logs ON urls(source, log_exported, id);
CREATE TABLE IF NOT EXISTS content_hashes (content_hash TEXT PRIMARY KEY, uid TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS switched (domain TEXT PRIMARY KEY, reason TEXT NOT NULL, at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS wayback (url_norm TEXT PRIMARY KEY, timestamp TEXT NOT NULL, original TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS export_batches (
    batch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source TEXT NOT NULL,
    kind TEXT NOT NULL,
    filename TEXT NOT NULL,
    ids_json TEXT NOT NULL,
    state TEXT NOT NULL DEFAULT 'prepared'
);
"""


def find_inputs(name: str) -> list[Path]:
    return sorted(INPUT_ROOT.rglob(name)) if INPUT_ROOT.is_dir() else []


def is_our_output(rel: PurePosixPath) -> bool:
    if len(rel.parts) != 5 or rel.parts[:2] != ("corpus", "viz") or rel.parts[3] != OWNER:
        return False
    kind, name = rel.parts[2], rel.parts[4]
    if kind in ("docs", "manifest"):
        return name.startswith(DOC_PREFIXES) and not name.endswith(".tmp")
    if kind == "crawl_log":
        return name.startswith(LOG_PREFIXES) and not name.endswith(".tmp")
    return False


def restore_checkpoint() -> None:
    if STATE_PATH.exists():
        print(f"Dùng checkpoint đang có trong thư mục làm việc: {STATE_PATH}")
        return
    bundles = sorted(find_inputs(BUNDLE_PATH.name), key=lambda p: p.stat().st_mtime, reverse=True)
    if bundles:
        restored = 0
        with zipfile.ZipFile(bundles[0]) as archive:
            for member in archive.infolist():
                rel = PurePosixPath(member.filename)
                if member.is_dir() or rel.is_absolute() or ".." in rel.parts:
                    continue
                if rel.as_posix() == STATE_PATH.name:
                    target = STATE_PATH
                elif is_our_output(rel):
                    target = WORK_ROOT.joinpath(*rel.parts)
                else:
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
                restored += 1
        if not STATE_PATH.exists():
            raise RuntimeError(f"Bundle {bundles[0]} không chứa {STATE_PATH.name}; không thể resume an toàn.")
        print(f"Đã khôi phục checkpoint từ ZIP: {bundles[0]} ({restored} file)")
        return
    states = find_inputs(STATE_PATH.name)
    if states:
        source_root = states[0].parent
        shutil.copy2(states[0], STATE_PATH)
        copied = 0
        source_viz = source_root / "corpus" / "viz"
        if source_viz.is_dir():
            for path in source_viz.rglob("*"):
                rel = PurePosixPath(path.relative_to(source_root).as_posix())
                if path.is_file() and is_our_output(rel):
                    target = WORK_ROOT.joinpath(*rel.parts)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(path, target)
                    copied += 1
        print(f"Đã khôi phục checkpoint đã giải nén: {states[0]} ({copied} file shard/log/manifest)")
        return
    print("Không có checkpoint đầu vào; bắt đầu từ đầu.")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def locate_links() -> Path:
    candidates = [LINKS_LOCAL, *find_inputs(LINKS_FILENAME), *find_inputs("links_corpus.parquet")]
    for candidate in candidates:
        if candidate.is_file() and sha256_file(candidate) == LINKS_SHA256:
            print(f"Đã xác minh file BTC (SHA-256 khớp): {candidate}")
            return candidate
    print("Không thấy file BTC đúng bản chính thức trong input; tải từ Hugging Face...")
    download = LINKS_LOCAL.with_name(LINKS_LOCAL.name + ".download")
    with urlopen(Request(LINKS_HF_URL, headers={"User-Agent": USER_AGENT}), timeout=300) as response, download.open("wb") as output:
        shutil.copyfileobj(response, output)
    actual = sha256_file(download)
    if actual != LINKS_SHA256:
        download.unlink(missing_ok=True)
        raise RuntimeError(f"File tải về không khớp SHA-256 bản chính thức của BTC: {actual}")
    download.replace(LINKS_LOCAL)
    return LINKS_LOCAL


def connect_state(path: Path) -> sqlite3.Connection:
    connection = sqlite3.connect(path, timeout=60)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA journal_mode=WAL")
    connection.execute("PRAGMA synchronous=NORMAL")
    connection.executescript(STATE_SCHEMA)
    return connection


def meta_get(conn: sqlite3.Connection, key: str) -> str | None:
    row = conn.execute("SELECT value FROM metadata WHERE key=?", (key,)).fetchone()
    return None if row is None else str(row["value"])


def meta_set(conn: sqlite3.Connection, key: str, value: str) -> None:
    conn.execute("INSERT OR REPLACE INTO metadata(key, value) VALUES(?, ?)", (key, value))


def strip_accents(value: str) -> str:
    value = unicodedata.normalize("NFD", value.lower()).replace("đ", "d")
    return "".join(c for c in value if unicodedata.category(c) != "Mn")


def url_slug(url: str) -> str:
    path = unquote(urlsplit(url).path)
    path = re.sub(r"\.(html?|htm|aspx?|php|ldo)$", "", path)
    path = re.sub(r"[-_/]?\d{4,}", " ", path)
    return strip_accents(re.sub(r"[-_/.]+", " ", path))


def priority_order(urls: list[str]) -> list[int]:
    """Thứ tự crawl: với mỗi truy vấn, xếp link theo độ gần giữa slug URL (tiêu đề bài) và truy vấn (TF-IDF, bỏ dấu);
    link được ưu tiên theo thứ hạng tốt nhất của nó qua mọi truy vấn → truy vấn nào cũng được lấy link khớp nhất trước.
    Không có file truy vấn thì xếp xen kẽ theo domain."""
    queries_files = find_inputs(QUERIES_FILENAME)
    if not queries_files:
        print(f"CẢNH BÁO: không thấy {QUERIES_FILENAME} trong input → không xếp ưu tiên theo truy vấn (kém hiệu quả hơn nhiều)")
        seen: Counter[str] = Counter()
        keys = []
        for url in urls:
            host = normalize_hostname(url) or ""
            keys.append(seen[host]); seen[host] += 1
        return sorted(range(len(urls)), key=lambda i: keys[i])
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer

    queries = pq.read_table(queries_files[0], columns=["query"]).column("query").to_pylist()
    print(f"Xếp ưu tiên theo {len(queries):,} truy vấn trong {queries_files[0]} ...")
    slugs = [url_slug(u) for u in urls]
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    X = vec.fit_transform(slugs)
    Q = vec.transform([strip_accents(str(q)) for q in queries])
    worst = np.iinfo(np.int64).max
    best_rank = np.full(len(urls), worst, dtype=np.int64)
    best_score = np.zeros(len(urls))
    k = min(1000, len(urls) - 1)
    for start in range(0, Q.shape[0], 50):
        scores = (Q[start:start + 50] @ X.T).toarray()
        best_score = np.maximum(best_score, scores.max(axis=0))
        for row in scores:
            top = np.argpartition(-row, k)[:k]
            top = top[row[top] > 0]
            top = top[np.argsort(-row[top])]
            best_rank[top] = np.minimum(best_rank[top], np.arange(len(top)))
    order = np.lexsort((-best_score, best_rank))
    ranked = int((best_rank < worst).sum())
    print(f"  {ranked:,} link nằm trong top-{k} của ít nhất một truy vấn (được crawl trước), {len(urls) - ranked:,} link xếp sau")
    return order.tolist()


def seed_urls(conn: sqlite3.Connection, links_path: Path) -> None:
    stored = meta_get(conn, "links_sha256")
    if stored is not None and stored != LINKS_SHA256:
        raise RuntimeError("Checkpoint thuộc snapshot links khác; không ghép trạng thái.")
    with conn:
        conn.execute("UPDATE urls SET status='pending' WHERE status='fetching'")
    if meta_get(conn, "seeded") == "1":
        skip_known_ids(conn)
        return
    table = pq.read_table(links_path, columns=["id", "url"])
    if table.num_rows != LINKS_ROWS:
        raise RuntimeError(f"File BTC có {table.num_rows:,} dòng, mong đợi {LINKS_ROWS:,}.")
    targets = set(TARGET_DOMAINS)
    picked = []
    for source_id, url in zip(table.column("id").to_pylist(), table.column("url").to_pylist()):
        host = normalize_hostname(url)
        if host in targets and int(source_id) not in ALREADY_CRAWLED_IDS:
            picked.append((int(source_id), url, host))
    order = priority_order([url for _, url, _ in picked])
    rows = []
    for rank, i in enumerate(order):
        source_id, url, host = picked[i]
        if host in WAYBACK_LAST:
            rank += len(order)
        rows.append((source_id, url, host, "wayback" if host in WAYBACK_FIRST else "direct", rank))
    with conn:
        conn.executemany("INSERT OR IGNORE INTO urls(id, url, domain, source, priority) VALUES(?, ?, ?, ?, ?)", rows)
        meta_set(conn, "links_sha256", LINKS_SHA256)
        meta_set(conn, "seeded", "1")
    by_lane = Counter(r[3] for r in rows)
    print(f"Đã nạp {len(rows):,} URL ({by_lane['direct']:,} trực tiếp, {by_lane['wayback']:,} qua Wayback)")


def url_norm(url: str) -> str:
    """Khóa so khớp URL BTC với URL trong chỉ mục CDX: bỏ scheme, www., query/fragment và '/' cuối; host chữ thường."""
    value = url.split("#", 1)[0].split("?", 1)[0].rstrip("/")
    value = re.sub(r"^[a-z]+://", "", value, flags=re.I)
    host, _, path = value.partition("/")
    host = host.lower().split(":", 1)[0]
    return f"{host.removeprefix('www.')}/{path}"


def cdx_page(prefix: str, resume_key: str | None) -> tuple[list[list[str]], str | None]:
    params = {
        "url": prefix, "matchType": "prefix", "output": "json", "fl": "original,timestamp",
        "filter": ["statuscode:200", "mimetype:text/html"], "collapse": "urlkey", "limit": "50000",
        "showResumeKey": "true",
    }
    if resume_key:
        params["resumeKey"] = resume_key
    from urllib.parse import urlencode
    from urllib.error import HTTPError, URLError
    url = f"{CDX_URL}?{urlencode(params, doseq=True)}"
    last_error = None
    for attempt in range(6):
        try:
            with urlopen(Request(url, headers={"User-Agent": USER_AGENT}), timeout=600) as response:
                rows = json.loads(response.read().decode("utf-8") or "[]")
            break
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            wait = 30 * (attempt + 1)
            print(f"  CDX lỗi ({type(exc).__name__}: {exc}); thử lại sau {wait} giây", flush=True)
            time.sleep(wait)
    else:
        raise RuntimeError(f"Không tra được chỉ mục Wayback CDX sau 6 lần: {last_error}")
    next_key = None
    if rows and len(rows[-1]) == 1:                       # dòng cuối là resumeKey, ngay trước nó là dòng rỗng
        next_key = rows[-1][0]
        rows = rows[:-1]
        while rows and not rows[-1]:
            rows = rows[:-1]
    if rows and rows[0] == ["original", "timestamp"]:
        rows = rows[1:]
    return rows, next_key


def index_cdx(prefix: str) -> None:
    """Tra chỉ mục CDX của một tiền tố (resume theo trang). Chạy trong thread riêng với kết nối SQLite riêng (WAL)."""
    conn = connect_state(STATE_PATH)
    conn.execute("PRAGMA busy_timeout=60000")
    try:
        done_key, page_key, resume_meta = f"cdx_done:{prefix}", f"cdx_pages:{prefix}", f"cdx_resume:{prefix}"
        if meta_get(conn, done_key) == "1":
            return
        page = int(meta_get(conn, page_key) or 0)
        resume_key = meta_get(conn, resume_meta)
        print(f"Tra chỉ mục Wayback CDX cho {prefix} (từ trang {page + 1}) ...", flush=True)
        while True:
            rows, next_key = cdx_page(prefix, resume_key)
            with conn:
                conn.executemany(
                    "INSERT OR IGNORE INTO wayback(url_norm, timestamp, original) VALUES(?, ?, ?)",
                    [(url_norm(original), ts, original) for original, ts in (r[:2] for r in rows if len(r) >= 2)],
                )
                page += 1
                meta_set(conn, page_key, str(page))
                if next_key:
                    meta_set(conn, resume_meta, next_key)
                else:
                    meta_set(conn, done_key, "1")
            print(f"  {prefix} trang {page}: {len(rows):,} bản lưu", flush=True)
            if not next_key:
                print(f"  {prefix}: tra xong chỉ mục CDX", flush=True)
                return
            resume_key = next_key
            time.sleep(5)
    finally:
        conn.close()


def snapshot_for(conn: sqlite3.Connection, url: str) -> tuple[str, str] | None | bool:
    """(timestamp, original) nếu có trong chỉ mục; None nếu tiền tố đã tra xong mà không có bản lưu; False nếu chưa tra."""
    key = url_norm(url)
    for prefix in CDX_PREFIXES.values():
        if key.startswith(prefix) and meta_get(conn, f"cdx_done:{prefix}") == "1":
            row = conn.execute("SELECT timestamp, original FROM wayback WHERE url_norm=?", (key,)).fetchone()
            return (row["timestamp"], row["original"]) if row else None
    return False


def skip_known_ids(conn: sqlite3.Connection) -> None:
    """Checkpoint tạo trước khi danh sách ALREADY_CRAWLED_IDS được cập nhật: bỏ các URL còn chờ mà nhóm đã có doc
    (không ghi doc, không ghi crawl log)."""
    with conn:
        conn.execute("CREATE TEMP TABLE IF NOT EXISTS known(id INTEGER PRIMARY KEY)")
        conn.executemany("INSERT OR IGNORE INTO known(id) VALUES(?)", [(i,) for i in ALREADY_CRAWLED_IDS])
        skipped = conn.execute(
            "UPDATE urls SET status='done', result='known', doc_exported=1, log_exported=1 "
            "WHERE status='pending' AND id IN (SELECT id FROM known)"
        ).rowcount
    if skipped:
        print(f"Bỏ qua {skipped:,} URL còn chờ vì nhóm đã có doc (shard của thành viên khác)")


def switch_to_wayback(conn: sqlite3.Connection, domain: str, reason: str) -> int:
    with conn:
        moved = conn.execute(
            "UPDATE urls SET source='wayback' WHERE source='direct' AND domain=? AND status='pending'", (domain,)
        ).rowcount
        conn.execute("INSERT OR REPLACE INTO switched(domain, reason, at) VALUES(?, ?, ?)", (domain, reason, timestamp_now()))
    print(f"\n{domain}: {reason} → chuyển {moved:,} URL còn lại sang Wayback")
    return moved


def claim(conn: sqlite3.Connection, lane: str, domain: str | None = None) -> sqlite3.Row | None:
    if domain is None:
        row = conn.execute(
            "SELECT id, url, domain FROM urls WHERE source=? AND status='pending' ORDER BY priority LIMIT 1", (lane,)
        ).fetchone()
    else:
        row = conn.execute(
            "SELECT id, url, domain FROM urls WHERE source=? AND domain=? AND status='pending' ORDER BY priority LIMIT 1",
            (lane, domain),
        ).fetchone()
    if row is not None:
        with conn:
            conn.execute("UPDATE urls SET status='fetching' WHERE id=?", (row["id"],))
    return row


def release(conn: sqlite3.Connection, ids: list[int]) -> None:
    with conn:
        conn.executemany("UPDATE urls SET status='pending' WHERE id=? AND status='fetching'", [(i,) for i in ids])


def save_results(conn: sqlite3.Connection, results: list[tuple[int, int, dict, dict | None]]) -> None:
    """Giống CrawlState.save_result: trùng content_hash → dup_of + status duplicate."""
    with conn:
        for source_id, attempts, log, doc in results:
            if doc is not None:
                original = conn.execute("SELECT uid FROM content_hashes WHERE content_hash=?", (doc["content_hash"],)).fetchone()
                if original is not None:
                    doc["dup_of"] = original["uid"]
                    doc["fetch"]["status"] = "duplicate"
                    log["status"] = "duplicate"
                else:
                    conn.execute("INSERT INTO content_hashes(content_hash, uid) VALUES(?, ?)", (doc["content_hash"], doc["uid"]))
            log["shard"] = None
            conn.execute(
                "UPDATE urls SET status='done', result=?, attempt=?, log_json=?, doc_json=? WHERE id=?",
                (
                    log["status"],
                    attempts,
                    json.dumps(log, ensure_ascii=False),
                    json.dumps(doc, ensure_ascii=False) if doc is not None else None,
                    source_id,
                ),
            )


def rows_for_ids(conn: sqlite3.Connection, ids: list[int]) -> list[sqlite3.Row]:
    rows: list[sqlite3.Row] = []
    for start in range(0, len(ids), 900):
        part = ids[start:start + 900]
        rows.extend(conn.execute(f"SELECT * FROM urls WHERE id IN ({','.join('?' for _ in part)})", part))
    return sorted(rows, key=lambda row: row["id"])


def next_sequence(conn: sqlite3.Connection, source: str, kind: str) -> int:
    if kind == "docs":
        directory, prefix, suffix = DOCS_DIR, f"viz-{OWNER}-{source}-", ".jsonl.zst"
    else:
        directory, prefix, suffix = LOGS_DIR, f"log-{OWNER}-{source}-", ".parquet"
    names = [p.name for p in directory.glob(f"{prefix}*{suffix}")]
    names += [r["filename"] for r in conn.execute("SELECT filename FROM export_batches WHERE source=? AND kind=?", (source, kind))]
    numbers = [int(n[len(prefix):-len(suffix)]) for n in names if n[len(prefix):-len(suffix)].isdigit()]
    return max(numbers, default=0) + 1


def extractor_for(source: str) -> str:
    version = f"trafilatura@{trafilatura.__version__}"
    return version + "+wayback-id_" if source == "wayback" else version


def write_docs(conn: sqlite3.Connection, source: str, filename: str, ids: list[int]) -> None:
    path = DOCS_DIR / filename
    manifest_path = MANIFESTS_DIR / f"{filename}.manifest.json"
    temp = path.with_suffix(path.suffix + ".tmp")
    temp_manifest = manifest_path.with_suffix(manifest_path.suffix + ".tmp")
    status_counts: Counter[str] = Counter()
    lang_counts: Counter[str] = Counter()
    with temp.open("wb") as raw:
        with zstandard.ZstdCompressor(level=3).stream_writer(raw, closefd=False) as compressed:
            for row in rows_for_ids(conn, ids):
                if row["doc_json"] is None:
                    continue
                doc = json.loads(row["doc_json"])
                doc["fetch"]["status"] = "duplicate" if doc["dup_of"] else doc["fetch"]["status"]
                validate_doc(doc, OWNER)
                status_counts[doc["fetch"]["status"]] += 1
                lang_counts[doc["lang"]] += 1
                compressed.write((json.dumps(doc, ensure_ascii=False) + "\n").encode("utf-8"))
        raw.flush()
        os.fsync(raw.fileno())
    count, digest = validate_shard_file(temp, OWNER)
    if count != sum(status_counts.values()):
        raise ValueError("Document count differs after writing the zstd shard")
    os.replace(temp, path)
    manifest = {
        "shard": path.name,
        "owner": OWNER,
        "schema_version": SCHEMA_VERSION,
        "norm_version": NORM_VERSION,
        "n_docs": count,
        "status_counts": dict(status_counts),
        "lang_counts": dict(lang_counts),
        "sha256": digest,
        "created_at": timestamp_now(),
        "input": MANIFEST_INPUT[source],
        "extractor": extractor_for(source),
        "code_commit": CODE_COMMIT,
        "source_ids": ids,
    }
    temp_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temp_manifest, manifest_path)


def write_logs(conn: sqlite3.Connection, filename: str, ids: list[int]) -> None:
    records = []
    for row in rows_for_ids(conn, ids):
        if row["log_json"] is None:
            continue
        record = json.loads(row["log_json"])
        if row["doc_shard"]:
            record["shard"] = row["doc_shard"]
        records.append(record)
    path = LOGS_DIR / filename
    temp = path.with_suffix(path.suffix + ".tmp")
    pq.write_table(pa.Table.from_pylist(records, schema=LOG_SCHEMA), temp, compression="zstd")
    os.replace(temp, path)


def finish_batch(conn: sqlite3.Connection, batch_id: int, kind: str, filename: str, ids: list[int]) -> None:
    with conn:
        if kind == "docs":
            # doc_json được xóa sau khi shard đã ghi xong để checkpoint gọn; shard là bản lưu chính thức.
            conn.executemany(
                "UPDATE urls SET doc_exported=1, doc_shard=?, doc_json=NULL WHERE id=?", [(filename, i) for i in ids]
            )
        else:
            conn.executemany("UPDATE urls SET log_exported=1 WHERE id=?", [(i,) for i in ids])
        conn.execute("UPDATE export_batches SET state='done' WHERE batch_id=?", (batch_id,))


def export_batch(conn: sqlite3.Connection, source: str, kind: str, ids: list[int]) -> None:
    seq = next_sequence(conn, source, kind)
    filename = f"viz-{OWNER}-{source}-{seq:05d}.jsonl.zst" if kind == "docs" else f"log-{OWNER}-{source}-{seq:05d}.parquet"
    with conn:
        batch_id = conn.execute(
            "INSERT INTO export_batches(source, kind, filename, ids_json) VALUES(?, ?, ?, ?)",
            (source, kind, filename, json.dumps(ids)),
        ).lastrowid
    if kind == "docs":
        write_docs(conn, source, filename, ids)
    else:
        write_logs(conn, filename, ids)
    finish_batch(conn, batch_id, kind, filename, ids)
    print(f"\nĐã xuất {filename} ({len(ids):,} {'document' if kind == 'docs' else 'dòng log'})")


def unexported_ids(conn: sqlite3.Connection, source: str, kind: str) -> list[int]:
    if kind == "docs":
        sql = "SELECT id FROM urls WHERE source=? AND status='done' AND doc_json IS NOT NULL AND doc_exported=0 ORDER BY id LIMIT ?"
    else:
        sql = "SELECT id FROM urls WHERE source=? AND status='done' AND log_exported=0 ORDER BY id LIMIT ?"
    return [int(row["id"]) for row in conn.execute(sql, (source, SHARD_DOCS))]


def export_ready(conn: sqlite3.Connection, source: str, force: bool = False) -> None:
    """Giống ShardWriter.export_ready: đủ SHARD_DOCS thì đóng shard; force=True xuất phần còn lại."""
    while True:
        doc_ids = unexported_ids(conn, source, "docs")
        if len(doc_ids) >= SHARD_DOCS or (force and doc_ids):
            export_batch(conn, source, "docs", doc_ids)
            continue
        log_ids = unexported_ids(conn, source, "logs")
        if len(log_ids) >= SHARD_DOCS or (force and log_ids):
            if doc_ids:
                export_batch(conn, source, "docs", doc_ids)
            export_batch(conn, source, "logs", log_ids)
            continue
        return


def recover_and_verify_exports(conn: sqlite3.Connection) -> None:
    for batch in conn.execute("SELECT * FROM export_batches WHERE state='prepared' ORDER BY batch_id").fetchall():
        ids = json.loads(batch["ids_json"])
        if batch["kind"] == "docs":
            write_docs(conn, batch["source"], batch["filename"], ids)
        else:
            write_logs(conn, batch["filename"], ids)
        finish_batch(conn, batch["batch_id"], batch["kind"], batch["filename"], ids)
        print(f"Đã hoàn tất lại lô xuất dang dở: {batch['filename']}")
    missing = []
    for batch in conn.execute("SELECT kind, filename FROM export_batches WHERE state='done'"):
        if batch["kind"] == "docs":
            expected = [DOCS_DIR / batch["filename"], MANIFESTS_DIR / f"{batch['filename']}.manifest.json"]
        else:
            expected = [LOGS_DIR / batch["filename"]]
        missing += [p.name for p in expected if not p.is_file()]
    if missing:
        raise RuntimeError(
            f"Checkpoint ghi nhận đã xuất nhưng thiếu {len(missing)} file (vd: {missing[:3]}). "
            f"Hãy đính kèm đúng bundle {BUNDLE_PATH.name} của phiên trước."
        )


def make_bundle(conn: sqlite3.Connection) -> None:
    snapshot = WORK_ROOT / f"{STATE_PATH.name}.snapshot"
    snapshot.unlink(missing_ok=True)
    target = sqlite3.connect(snapshot)
    try:
        conn.backup(target)
    finally:
        target.close()
    temp = BUNDLE_PATH.with_name(BUNDLE_PATH.name + ".tmp")
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(snapshot, arcname=STATE_PATH.name)
        for directory in (DOCS_DIR, LOGS_DIR, MANIFESTS_DIR):
            for path in sorted(directory.iterdir()):
                rel = PurePosixPath(path.relative_to(WORK_ROOT).as_posix())
                if path.is_file() and is_our_output(rel):
                    stored = path.suffix in {".zst", ".parquet"}
                    archive.write(path, arcname=rel.as_posix(), compress_type=zipfile.ZIP_STORED if stored else zipfile.ZIP_DEFLATED)
    os.replace(temp, BUNDLE_PATH)
    snapshot.unlink(missing_ok=True)
    print(f"\nĐã lưu bundle resume: {BUNDLE_PATH} ({BUNDLE_PATH.stat().st_size / 1e6:.1f} MB)")


restore_checkpoint()
LINKS_PATH = locate_links()
conn = connect_state(STATE_PATH)
recover_and_verify_exports(conn)
seed_urls(conn, LINKS_PATH)
for _row in conn.execute("SELECT source, COUNT(*) AS total, SUM(status='done') AS done FROM urls GROUP BY source"):
    print(f"  {_row['source']}: {_row['done'] or 0:,}/{_row['total']:,} đã xử lý")

In [ ]:
# Tải trang: luồng "direct" (từng domain một cổng giãn cách riêng, tuân thủ robots.txt ở mọi bước chuyển hướng) và luồng
# "wayback" (bản lưu id_ trên web.archive.org). Domain chặn crawler thì tự chuyển sang Wayback — không giả trình duyệt,
# không vượt trang kiểm tra bot.


import threading

# langdetect nạp hồ sơ ngôn ngữ lười ở lần gọi đầu; nhiều thread gọi đồng thời lúc đó cho kết quả sai (bài tiếng Việt bị
# nhận là "en", gặp khi chạy thử 28/10). Nạp trước một lần ở thread chính và khóa mọi lần gọi sau.
_LANG_LOCK = threading.Lock()
_detect_language_unlocked = detect_supported_language


def detect_supported_language(text: str) -> tuple[str, float] | None:
    with _LANG_LOCK:
        return _detect_language_unlocked(text)


assert detect_supported_language("Bệnh tiểu đường là bệnh mạn tính, cần điều trị và theo dõi lâu dài.")[0] == "vi"


@dataclass
class FetchResult:
    status: str
    http_status: int | None
    fetched_at: str
    final_url: str | None
    body: bytes | None
    error: str | None
    attempts: int
    nbytes: int | None
    encoding: str | None = None
    transient: bool = False                 # lỗi tạm thời (429/5xx/timeout): trả URL về hàng chờ thay vì đánh dấu xong
    block_like: bool = False                # 401/403/trang kiểm tra bot: đếm để chuyển domain sang Wayback


class AdaptiveGate:
    def __init__(self, min_delay: float, max_delay: float) -> None:
        self.min_delay, self.max_delay, self.delay = min_delay, max(max_delay, min_delay), min_delay
        self.next_slot = 0.0
        self.ok_streak = 0
        self.lock = asyncio.Lock()

    async def wait(self) -> None:
        async with self.lock:
            now = time.monotonic()
            if self.next_slot > now:
                await asyncio.sleep(self.next_slot - now)
            self.next_slot = time.monotonic() + self.delay

    def slow_down(self, retry_after: float | None) -> None:
        self.delay = min(self.max_delay, self.delay * 2)
        self.ok_streak = 0
        self.next_slot = max(self.next_slot, time.monotonic() + (retry_after if retry_after is not None else 30.0))

    def success(self) -> None:
        self.ok_streak += 1
        if self.ok_streak >= 200 and self.delay > self.min_delay:
            self.delay = max(self.min_delay, self.delay / 1.5)
            self.ok_streak = 0


WAYBACK_GATE = AdaptiveGate(WAYBACK_MIN_DELAY_SECONDS, WAYBACK_MAX_DELAY_SECONDS)
DOMAIN_GATES: dict[str, AdaptiveGate] = {}
REQUEST_HEADERS = {"User-Agent": USER_AGENT, "Accept": "text/html,application/xhtml+xml;q=0.9,*/*;q=0.8",
                   "Accept-Language": "vi,en;q=0.5"}
_CHALLENGE_MARKERS = (b"captcha", b"verify you are human", b"just a moment...", b"cf-chl", b"challenge-platform")


def parse_retry_after(value: str | None) -> float | None:
    if not value:
        return None
    try:
        return max(0.0, float(value))
    except ValueError:
        try:
            parsed = parsedate_to_datetime(value)
            return max(0.0, (parsed - datetime.now(parsed.tzinfo)).total_seconds())
        except (TypeError, ValueError, OverflowError):
            return None


async def read_body(response: aiohttp.ClientResponse, limit: int) -> bytes:
    # StreamReader.read(n) chỉ trả phần đang có trong buffer (vd 12KB của trang 800KB) nên phải đọc đến EOF.
    chunks: list[bytes] = []
    size = 0
    async for chunk in response.content.iter_chunked(64 * 1024):
        chunks.append(chunk)
        size += len(chunk)
        if size > limit:
            break
    return b"".join(chunks)


def is_challenge(body: bytes) -> bool:
    head = body[:8192].lower()
    if any(marker in head for marker in _CHALLENGE_MARKERS):
        return True
    # trang chỉ có script đặt cookie rồi tải lại (vd laodong.vn) — không phải bài viết
    return len(body) < 4096 and b"document.cookie" in head and b"reload" in head


class RobotsCache:
    """robots.txt theo từng scheme+host. None = không đọc được (401/403/5xx/lỗi mạng) → không tải trực tiếp host đó."""

    def __init__(self) -> None:
        self.parsers: dict[str, Protego | None] = {}
        self.errors: dict[str, str] = {}
        self.locks: dict[str, asyncio.Lock] = {}

    async def get(self, session: aiohttp.ClientSession, url: str) -> Protego | None:
        parts = urlsplit(url)
        key = f"{parts.scheme}://{parts.netloc.lower()}"
        lock = self.locks.setdefault(key, asyncio.Lock())
        async with lock:
            if key not in self.parsers:
                self.parsers[key] = await self._load(session, key)
        return self.parsers[key]

    async def _load(self, session: aiohttp.ClientSession, key: str) -> Protego | None:
        for attempt in range(3):
            try:
                async with session.get(f"{key}/robots.txt", headers=REQUEST_HEADERS, allow_redirects=True,
                                       timeout=aiohttp.ClientTimeout(total=HTTP_TIMEOUT_SECONDS)) as response:
                    if response.status in (404, 410):
                        return Protego.parse("")
                    if response.status == 200:
                        content = (await read_body(response, 512_000))[:512_000].decode("utf-8", errors="replace")
                        return Protego.parse(content)
                    self.errors[key] = f"robots.txt HTTP {response.status}"
                    if response.status < 500 and response.status != 429:
                        return None
            except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                self.errors[key] = f"robots.txt lỗi {type(exc).__name__}"
            await asyncio.sleep(5 * (attempt + 1))
        return None


ROBOTS = RobotsCache()


def _path_is_home(url: str) -> bool:
    return urlsplit(url).path.strip("/") in ("", "index.html", "index.htm", "home")


async def fetch_direct(session: aiohttp.ClientSession, gate: AdaptiveGate, url: str) -> FetchResult:
    attempts, last_error = 0, None
    for attempt in range(MAX_RETRIES + 1):
        attempts += 1
        current, redirects, retry = url, 0, False
        while True:
            parser = await ROBOTS.get(session, current)
            if parser is None:
                return FetchResult("blocked", None, timestamp_now(), current, None,
                                   f"Không đọc được robots.txt của {urlsplit(current).netloc}", attempts, None, block_like=True)
            if not parser.can_fetch(current, USER_AGENT):
                return FetchResult("blocked", None, timestamp_now(), current, None, "Disallowed by robots.txt", attempts, None)
            await gate.wait()
            try:
                async with session.get(current, headers=REQUEST_HEADERS, allow_redirects=False,
                                       timeout=aiohttp.ClientTimeout(total=HTTP_TIMEOUT_SECONDS)) as response:
                    status = response.status
                    if status in {301, 302, 303, 307, 308}:
                        location = response.headers.get("Location")
                        target = urljoin(str(response.url), location) if location else None
                        if not target or redirects >= 5:          # vòng chuyển hướng ở mọi trang = không vào được
                            return FetchResult("blocked", status, timestamp_now(), target, None,
                                               "Chuyển hướng hỏng hoặc quá 5 lần", attempts, None, block_like=True)
                        if _path_is_home(target) and not _path_is_home(url):
                            return FetchResult("http_error", status, timestamp_now(), target, None,
                                               "Bài không còn (chuyển về trang chủ)", attempts, None)
                        current, redirects = target, redirects + 1
                        continue
                    body = await read_body(response, MAX_RESPONSE_BYTES)
                    fetched_at, final_url = timestamp_now(), str(response.url)
                    if status in {429, 503}:
                        last_error = f"HTTP {status}"
                        gate.slow_down(parse_retry_after(response.headers.get("Retry-After")))
                        retry = True
                        break
                    if status >= 500 or status == 408:
                        last_error = f"HTTP {status}"
                        retry = True
                        break
                    if status in {401, 403, 451}:
                        return FetchResult("blocked", status, fetched_at, final_url, None, f"HTTP {status}", attempts,
                                           len(body), block_like=True)
                    if status >= 400:
                        return FetchResult("http_error", status, fetched_at, final_url, None, f"HTTP {status}", attempts,
                                           len(body), response.charset)
                    if len(body) > MAX_RESPONSE_BYTES:
                        return FetchResult("http_error", status, fetched_at, final_url, None,
                                           "Response exceeds configured size limit", attempts, len(body), response.charset)
                    if is_challenge(body):
                        return FetchResult("blocked", status, fetched_at, final_url, None,
                                           "Trang kiểm tra bot, không phải bài viết", attempts, len(body), block_like=True)
                    gate.success()
                    return FetchResult("ok", status, fetched_at, final_url, body, None, attempts, len(body), response.charset)
            except asyncio.TimeoutError:
                last_error = "Request timeout"
                retry = True
            except aiohttp.ClientError as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                retry = True
            break
        if not retry:
            break
        if attempt < MAX_RETRIES:
            await asyncio.sleep(RETRY_BACKOFF_SECONDS * 2 ** attempt)
    status = "timeout" if last_error == "Request timeout" else "http_error"
    return FetchResult(status, None, timestamp_now(), None, None, last_error, attempts, None, transient=True)


_WAYBACK_ORIGINAL = re.compile(r"^/web/\d+id_/(.+)$")


async def fetch_wayback(session: aiohttp.ClientSession, parser: Protego, url: str,
                        snapshot: tuple[str, str] | None = None) -> FetchResult:
    if snapshot is not None:                   # đã biết đúng bản lưu từ chỉ mục CDX → tải thẳng, không qua chuyển hướng
        start_url = f"https://{WAYBACK_HOST}/web/{snapshot[0]}id_/{snapshot[1]}"
    else:
        start_url = f"https://{WAYBACK_HOST}/web/{WAYBACK_TIMESTAMP}id_/{url}"
    attempts, last_error = 0, None
    for attempt in range(MAX_RETRIES + 1):
        attempts += 1
        current_url, redirects, retry = start_url, 0, False
        while True:
            await WAYBACK_GATE.wait()
            try:
                async with session.get(current_url, headers=REQUEST_HEADERS, allow_redirects=False,
                                       timeout=aiohttp.ClientTimeout(total=HTTP_TIMEOUT_SECONDS)) as response:
                    if response.status in {301, 302, 303, 307, 308}:
                        location = response.headers.get("Location")
                        target = urljoin(str(response.url), location) if location else None
                        if not target or redirects >= 6 or urlsplit(target).hostname != WAYBACK_HOST:
                            return FetchResult("http_error", response.status, timestamp_now(), target, None,
                                               "Redirect ra ngoài web.archive.org hoặc quá 6 lần", attempts, None)
                        if not parser.can_fetch(target, USER_AGENT):
                            return FetchResult("blocked", response.status, timestamp_now(), target, None,
                                               "Disallowed by web.archive.org robots.txt", attempts, None)
                        original = _WAYBACK_ORIGINAL.match(urlsplit(target).path)
                        if original and _path_is_home(original.group(1)) and not _path_is_home(url):
                            return FetchResult("http_error", response.status, timestamp_now(), target, None,
                                               "Bản lưu chuyển về trang chủ (bài không còn)", attempts, None)
                        current_url, redirects = target, redirects + 1
                        continue
                    body = await read_body(response, MAX_RESPONSE_BYTES)
                    fetched_at, final_url = timestamp_now(), str(response.url)
                    if response.status in {429, 503}:
                        last_error = f"HTTP {response.status}"
                        WAYBACK_GATE.slow_down(parse_retry_after(response.headers.get("Retry-After")))
                        retry = True
                        break
                    if response.status >= 500 or response.status == 408:
                        last_error = f"HTTP {response.status}"
                        retry = True
                        break
                    if response.status == 404:
                        return FetchResult("http_error", 404, fetched_at, final_url, None,
                                           "Không có bản lưu Wayback", attempts, len(body))
                    if response.status >= 400:
                        return FetchResult("http_error", response.status, fetched_at, final_url, None,
                                           f"Wayback HTTP {response.status}", attempts, len(body), response.charset)
                    if len(body) > MAX_RESPONSE_BYTES:
                        return FetchResult("http_error", response.status, fetched_at, final_url, None,
                                           "Response exceeds configured size limit", attempts, len(body), response.charset)
                    if is_challenge(body):
                        return FetchResult("blocked", response.status, fetched_at, final_url, None,
                                           "Bản lưu là trang kiểm tra truy cập, không phải bài viết", attempts, len(body))
                    WAYBACK_GATE.success()
                    return FetchResult("ok", response.status, fetched_at, final_url, body, None, attempts, len(body),
                                       response.charset)
            except asyncio.TimeoutError:
                last_error = "Request timeout"
                retry = True
            except aiohttp.ClientError as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                retry = True
            break
        if not retry:
            break
        if attempt < MAX_RETRIES:
            await asyncio.sleep(RETRY_BACKOFF_SECONDS * 2 ** attempt)
    status = "timeout" if last_error == "Request timeout" else "http_error"
    return FetchResult(status, None, timestamp_now(), None, None, last_error, attempts, None, transient=True)


def extract_html(body: bytes, encoding: str | None) -> tuple[str, str]:
    try:
        downloaded = body.decode(encoding or "utf-8", errors="replace")
    except LookupError:
        downloaded = body.decode("utf-8", errors="replace")
    article = trafilatura.extract(
        downloaded,
        include_comments=False,
        include_tables=True,
        include_links=False,
        favor_precision=False,
        output_format="txt",
    )
    metadata = trafilatura.extract_metadata(downloaded)
    title = (metadata.title or "") if metadata else ""
    return re.sub(r"\s*[|•]\s*[^|•]{1,40}$", "", title), article or ""


def build_result(lane: str, row: sqlite3.Row, result: FetchResult) -> tuple[dict, dict | None]:
    raw_title = raw_body = None
    if result.status == "ok" and result.body is not None:
        raw_title, raw_body = extract_html(result.body, result.encoding)
    return build_records(
        source=lane, source_id=int(row["id"]), url=row["url"], domain=row["domain"], fetch_status=result.status,
        http_status=result.http_status, fetched_at=result.fetched_at, final_url=result.final_url,
        attempts=result.attempts, nbytes=result.nbytes, error=result.error, raw_title=raw_title, raw_body=raw_body,
        doc_url=row["url"], body_type="body", extractor=extractor_for(lane), pubmed_meta=None,
    )


async def handle(conn: sqlite3.Connection, lane: str, row: sqlite3.Row, result: FetchResult, stats: Counter,
                 domain_stats: Counter) -> None:
    log, doc = await asyncio.to_thread(build_result, lane, row, result)
    save_results(conn, [(int(row["id"]), result.attempts, log, doc)])
    stats[log["status"]] += 1
    domain_stats[row["domain"]] += 1
    export_ready(conn, lane)


async def run_direct_domain(conn: sqlite3.Connection, session: aiohttp.ClientSession, domain: str, deadline: float,
                            stats: Counter, domain_stats: Counter) -> str | None:
    first = conn.execute("SELECT url FROM urls WHERE source='direct' AND domain=? AND status='pending' LIMIT 1",
                         (domain,)).fetchone()
    if first is None:
        return None
    parser = await ROBOTS.get(session, first["url"])
    if parser is None:
        key = f"{urlsplit(first['url']).scheme}://{urlsplit(first['url']).netloc.lower()}"
        switch_to_wayback(conn, domain, ROBOTS.errors.get(key, "không đọc được robots.txt"))
        return "chuyển Wayback"
    delay = max(DIRECT_MIN_DELAY_SECONDS, float(parser.crawl_delay(USER_AGENT) or 0))
    gate = DOMAIN_GATES.setdefault(domain, AdaptiveGate(delay, max(DIRECT_MAX_DELAY_SECONDS, delay)))
    shared = {"claimed": 0, "transient": 0, "stopped": None}
    held: list[tuple[sqlite3.Row, FetchResult]] = []   # chuỗi bị chặn liên tiếp: chờ xem cả domain chặn hay chỉ vài trang

    async def worker() -> None:
        while shared["stopped"] is None and time.monotonic() < deadline:
            if MAX_FETCHES_PER_LANE is not None and shared["claimed"] >= MAX_FETCHES_PER_LANE:
                return
            row = claim(conn, "direct", domain)
            if row is None:
                return
            shared["claimed"] += 1
            result = await fetch_direct(session, gate, row["url"])
            if result.transient:
                release(conn, [int(row["id"])])
                shared["claimed"] -= 1
                shared["transient"] += 1
                stats["tạm_lỗi"] += 1
                if shared["transient"] >= MAX_CONSECUTIVE_TRANSIENT_FAILURES and shared["stopped"] is None:
                    shared["stopped"] = f"{shared['transient']} lỗi tạm thời liên tiếp ({result.error})"
                    return
                await asyncio.sleep(min(300, 10 * shared["transient"]))
                continue
            shared["transient"] = 0
            if result.block_like:
                held.append((row, result))
                if len(held) >= BLOCK_SWITCH_AFTER and shared["stopped"] is None:
                    shared["stopped"] = f"{len(held)} lần liên tiếp bị chặn ({result.error})"
                return_ids = [int(r["id"]) for r, _ in held] if shared["stopped"] else []
                if return_ids:
                    release(conn, return_ids)          # các URL này sẽ đi Wayback cùng phần còn lại của domain
                    held.clear()
                    return
                continue
            for held_row, held_result in held:         # chỉ vài trang bị chặn, domain vẫn vào được → ghi nhận bình thường
                await handle(conn, "direct", held_row, held_result, stats, domain_stats)
            held.clear()
            await handle(conn, "direct", row, result, stats, domain_stats)

    await asyncio.gather(*(worker() for _ in range(DIRECT_WORKERS_PER_DOMAIN)))
    if held:
        release(conn, [int(r["id"]) for r, _ in held])  # hết giờ khi đang có chuỗi bị chặn: để phiên sau quyết định
        held.clear()
    if shared["stopped"]:
        switch_to_wayback(conn, domain, shared["stopped"])
        return "chuyển Wayback"
    return None


async def run_wayback(conn: sqlite3.Connection, session: aiohttp.ClientSession, deadline: float, stats: Counter,
                      domain_stats: Counter, direct_running: dict) -> str | None:
    parser = await ROBOTS.get(session, f"https://{WAYBACK_HOST}/")
    if parser is None:
        return f"không xác minh được robots.txt của {WAYBACK_HOST}; URL giữ nguyên để resume"
    shared = {"claimed": 0, "transient": 0, "stopped": None}
    pending_domains = {r["domain"] for r in conn.execute("SELECT DISTINCT domain FROM urls WHERE source='wayback' AND status='pending'")}
    # chạy nền: trong lúc tra chỉ mục, URL của tiền tố đó vẫn được tải theo cách chuyển hướng (2 request)
    indexers = [asyncio.create_task(asyncio.to_thread(index_cdx, prefix))
                for domain, prefix in CDX_PREFIXES.items() if domain in pending_domains]

    async def worker() -> None:
        while shared["stopped"] is None and time.monotonic() < deadline:
            if MAX_FETCHES_PER_LANE is not None and shared["claimed"] >= MAX_FETCHES_PER_LANE:
                return
            row = claim(conn, "wayback")
            if row is None:
                if direct_running["n"] > 0:          # domain trực tiếp có thể còn chuyển URL sang Wayback
                    await asyncio.sleep(30)
                    continue
                return
            shared["claimed"] += 1
            snapshot = snapshot_for(conn, row["url"])
            if snapshot is None:
                result = FetchResult("http_error", None, timestamp_now(), None, None,
                                     "Không có bản lưu Wayback (theo chỉ mục CDX)", 0, None)
                await handle(conn, "wayback", row, result, stats, domain_stats)
                continue
            result = await fetch_wayback(session, parser, row["url"], snapshot or None)
            if result.transient:
                release(conn, [int(row["id"])])
                shared["claimed"] -= 1
                shared["transient"] += 1
                stats["tạm_lỗi"] += 1
                if shared["transient"] >= MAX_CONSECUTIVE_TRANSIENT_FAILURES:
                    shared["stopped"] = f"{shared['transient']} lỗi tạm thời liên tiếp ({result.error}); dừng để resume sau"
                    return
                await asyncio.sleep(min(300, 10 * shared["transient"]))
                continue
            shared["transient"] = 0
            await handle(conn, "wayback", row, result, stats, domain_stats)

    try:
        await asyncio.gather(*(worker() for _ in range(WAYBACK_WORKERS)))
    finally:
        for task in indexers:
            if not task.done():
                print("Chỉ mục CDX chưa tra xong; phiên sau tra tiếp từ trang đã dừng.")
        # thread tra CDX không hủy được giữa chừng; mỗi trang được ghi ngay nên phiên sau tiếp tục đúng chỗ
    for task in indexers:
        if task.done() and task.exception() is not None:
            print(f"Tra CDX lỗi (đã dùng cách chuyển hướng thay thế): {task.exception()}")
    return shared["stopped"]

In [ ]:
# Chạy đến khi xong hoặc hết MAX_RUNTIME_HOURS; luôn xuất shard + bundle khi dừng.


def print_progress(conn: sqlite3.Connection, stats: dict[str, Counter], domain_stats: Counter, started: float) -> None:
    elapsed = max(time.monotonic() - started, 1.0)
    now = time.strftime("%H:%M:%S")
    for row in conn.execute(
        "SELECT source, COUNT(*) AS total, SUM(status='done') AS done FROM urls GROUP BY source ORDER BY source"
    ):
        lane = row["source"]
        session_done = sum(v for k, v in stats[lane].items() if k != "tạm_lỗi")
        detail = ", ".join(f"{key}={value:,}" for key, value in sorted(stats[lane].items()))
        print(f"[{now}] {lane}: {row['done'] or 0:,}/{row['total']:,} | phiên này {session_done:,} "
              f"({session_done / elapsed:.2f} URL/s) | {detail}", flush=True)
    ok = dict(conn.execute("SELECT domain, COUNT(*) FROM urls WHERE result='ok' GROUP BY domain").fetchall())
    busy = ", ".join(f"{d} {n:,} (ok {ok.get(d, 0):,})" for d, n in domain_stats.most_common(8))
    if busy:
        print(f"           theo domain (phiên này): {busy}", flush=True)


def print_summary(conn: sqlite3.Connection) -> None:
    print("\n===== Tổng kết =====")
    for row in conn.execute(
        "SELECT source, COUNT(*) AS total, SUM(status='done') AS done, SUM(doc_exported) AS docs FROM urls GROUP BY source ORDER BY source"
    ):
        results = dict(conn.execute(
            "SELECT result, COUNT(*) FROM urls WHERE source=? AND status='done' GROUP BY result", (row["source"],)
        ).fetchall())
        print(f"{row['source']}: đã xử lý {row['done'] or 0:,}/{row['total']:,} | còn {row['total'] - (row['done'] or 0):,} | "
              f"document đã vào shard {row['docs'] or 0:,} | {results}")
    print("Document 'ok' theo domain:")
    for row in conn.execute(
        "SELECT domain, COUNT(*) AS total, SUM(result='ok') AS ok, SUM(status='done') AS done FROM urls "
        "GROUP BY domain ORDER BY total DESC"
    ):
        print(f"  {row['domain']:30} ok {row['ok'] or 0:>7,} | đã xử lý {row['done'] or 0:>7,}/{row['total']:,}")
    for row in conn.execute("SELECT * FROM switched ORDER BY at"):
        print(f"  (đã chuyển Wayback) {row['domain']}: {row['reason']}")
    shards = sorted(p.name for p in DOCS_DIR.glob("*.jsonl.zst") if p.name.startswith(DOC_PREFIXES))
    print(f"Shard document ({len(shards)}): {shards}")


async def run_all(conn: sqlite3.Connection) -> None:
    deadline = RUN_STARTED + MAX_RUNTIME_HOURS * 3600
    stats = {lane: Counter() for lane in LANES}
    domain_stats: Counter = Counter()
    started = time.monotonic()
    domains = [r["domain"] for r in conn.execute(
        "SELECT domain, MIN(priority) AS p FROM urls WHERE source='direct' AND status='pending' GROUP BY domain ORDER BY p")]
    direct_running = {"n": len(domains)}
    connector = aiohttp.TCPConnector(limit=2 * DIRECT_WORKERS_PER_DOMAIN * max(1, len(domains)) + WAYBACK_WORKERS + 4,
                                     limit_per_host=max(DIRECT_WORKERS_PER_DOMAIN, WAYBACK_WORKERS) + 1)
    async with aiohttp.ClientSession(connector=connector) as session:
        async def direct_job(domain: str) -> str | None:
            try:
                return await run_direct_domain(conn, session, domain, deadline, stats["direct"], domain_stats)
            finally:
                direct_running["n"] -= 1

        jobs = {f"direct:{d}": asyncio.create_task(direct_job(d)) for d in domains}
        jobs["wayback"] = asyncio.create_task(run_wayback(conn, session, deadline, stats["wayback"], domain_stats, direct_running))
        print(f"Bắt đầu: {len(domains)} domain tải trực tiếp + luồng Wayback")

        async def heartbeat() -> None:
            last_bundle = time.monotonic()
            while True:
                await asyncio.sleep(PROGRESS_SECONDS)
                print_progress(conn, stats, domain_stats, started)
                if time.monotonic() - last_bundle >= BUNDLE_EVERY_MINUTES * 60:
                    make_bundle(conn)
                    last_bundle = time.monotonic()

        reporter = asyncio.create_task(heartbeat())
        try:
            outcomes = await asyncio.gather(*jobs.values(), return_exceptions=True)
        finally:
            reporter.cancel()
            for job in jobs.values():
                job.cancel()
            await asyncio.gather(reporter, *jobs.values(), return_exceptions=True)
    print_progress(conn, stats, domain_stats, started)
    errors = []
    for name, outcome in zip(jobs, outcomes):
        if isinstance(outcome, BaseException):
            errors.append((name, outcome))
            print(f"{name}: LỖI {type(outcome).__name__}: {outcome}")
        elif outcome:
            print(f"{name}: {outcome}")
    if time.monotonic() >= deadline:
        print(f"Hết {MAX_RUNTIME_HOURS} giờ; dừng mềm, URL còn lại giữ để resume")
    if errors:
        raise errors[0][1]


try:
    await run_all(conn)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\nĐã nhận lệnh dừng; URL đang xử lý được đưa về chờ để resume.")
finally:
    with conn:
        conn.execute("UPDATE urls SET status='pending' WHERE status='fetching'")
    for _lane in LANES:
        export_ready(conn, _lane, force=True)
    print_summary(conn)
    make_bundle(conn)
    conn.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    conn.close()